# SME Legal QA — 2K Submission Batch Run (Colab A100)

**Goal**: Process the full competition test set (2000 questions) and produce
`results.json` + `submission.zip` in the grader format.

**Optimizations**:
- GPU-efficient batch processing (FAISS GPU, reranker batch=64)
- Streaming/incremental write of `results.json` (flush every 100 → no OOM,
  crash-safe — a partially written file is still inspectable)
- Pre-load all indices once

**Input**: `data/stage6_data/R2AIStage1DATA.json` — a JSON list of
`{"id", "question"}` records (the 2000-question test set).

**Output** (grader contract, per `ABOUT.md`):
- `results.json` — JSON array; each record has
  `id, question, answer, relevant_docs, relevant_articles`.
- `submission.zip` — `results.json` at the zip root, ready to upload.


In [ ]:
# ============================================================================
# 1. Environment setup (Colab A100)
# ============================================================================
!nvidia-smi

# Pin transformers + FlagEmbedding to a known-good combo. Newer transformers
# (>=4.46) break FlagEmbedding's BGE-m3 path: the slow XLMRobertaTokenizer loses
# `prepare_for_model` -> AttributeError on the first encode/compute_score call.
# faiss-gpu-cu12 covers Colab's CUDA 12; falls back to faiss-cpu otherwise.
!pip install -q faiss-gpu-cu12 'FlagEmbedding>=1.2.10,<1.3' 'transformers>=4.41,<4.46' accelerate pyarrow pyyaml regex tqdm networkx || pip install -q faiss-cpu 'FlagEmbedding>=1.2.10,<1.3' 'transformers>=4.41,<4.46' accelerate pyarrow pyyaml regex tqdm networkx

# If a newer transformers was already imported (Colab preloads one), the pins
# above only take effect after a kernel restart. Detect and force it.
import importlib
try:
    import transformers
    _tv = tuple(int(x) for x in transformers.__version__.split('.')[:3])
    if _tv >= (4, 46, 0):
        import os
        print(f"⚠ transformers {transformers.__version__} is loaded but >=4.46 breaks FlagEmbedding.")
        print("   Restarting kernel to load the pinned <4.46 version; this cell re-runs automatically.")
        os.kill(os.getpid(), 9)  # hard-restart the kernel; Colab re-executes the cell
except Exception as _e:
    print("(transformers version check skipped:", _e, ")")

# Mount Google Drive (assumes your data artifacts are in Drive)
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/Road2AI_ApplePie/src')


In [ ]:
# ============================================================================
# 2. Load artifacts (once)
# ============================================================================
import json
import pickle
import sqlite3
from pathlib import Path

import faiss
import networkx as nx
import pandas as pd
import yaml
from tqdm.auto import tqdm

# Adjust paths to your Drive structure
BASE = Path('/content/drive/MyDrive/Road2AI_ApplePie')
DATA = BASE / 'data/stage6_data'

# Config
with open(BASE / 'config/default.yaml') as f:
    cfg = yaml.safe_load(f)

# BM25 index (SQLite FTS5)
bm25_conn = sqlite3.connect(DATA / 'chunk_store.sqlite')
print(f"✓ BM25 index: {bm25_conn.execute('SELECT COUNT(*) FROM chunks_fts').fetchone()[0]} chunks")

# Chunk metadata (row_idx → law_id, dieu_so, etc.)
meta_df = pd.read_parquet(DATA / 'chunk_meta_slim.parquet')
print(f"✓ Metadata: {len(meta_df)} rows")

# FAISS index (GPU)
faiss_cpu = faiss.read_index(str(DATA / 'faiss_index__BAAI_bge-m3.index'))
res = faiss.StandardGpuResources()
faiss_gpu = faiss.index_cpu_to_gpu(res, 0, faiss_cpu)
print(f"✓ FAISS (GPU): {faiss_gpu.ntotal} vectors")

# NetworkX graph
with open(BASE / 'data/kg.gpickle', 'rb') as f:
    G = pickle.load(f)
print(f"✓ Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# row_to_uid mapping for GraphExpander
row_to_uid = dict(zip(meta_df['row_idx'], meta_df['doc_uid']))

# ---- PERF-CRITICAL: prebuild O(1) lookup dicts ----
# Repeated meta_df[meta_df.row_idx==x] / meta_df.iloc[x] scans during 2000×70
# lookups would add HOURS. Build dicts once.
#
# 1) FAISS position → metadata. The Stage 6 invariant is that FAISS vector
#    position i corresponds to meta_df row i (row_idx == i, contiguous). We
#    map by positional order to be robust, keyed by FAISS idx.
meta_records = meta_df.to_dict('records')
pos_to_meta = {i: rec for i, rec in enumerate(meta_records)}
row_to_meta = {int(rec['row_idx']): rec for rec in meta_records}

# 2) row_idx → chunk_text, pulled ONCE from the SQLite chunk store (text lives
#    there, not in the slim parquet). One bulk query → dict.
text_by_row = {}
try:
    cur = bm25_conn.execute('SELECT row_idx, chunk_text FROM chunks')
    for ridx, txt in cur.fetchall():
        text_by_row[int(ridx)] = txt or ''
    print(f"✓ Text cache: {len(text_by_row)} chunks (from chunks table)")
except Exception as e:
    # Fallback: text column may live in the FTS table or the parquet.
    print(f"⚠ chunks.text query failed ({e}); trying parquet column")
    if 'chunk_text' in meta_df.columns:
        text_by_row = {int(r['row_idx']): (r.get('chunk_text') or '')
                       for r in meta_records}
        print(f"✓ Text cache from parquet: {len(text_by_row)} chunks")

del faiss_cpu  # free CPU copy

In [ ]:
# ============================================================================
# 3. Load models (embedding + reranker, GPU batch mode)
# ============================================================================
from FlagEmbedding import BGEM3FlagModel, FlagReranker

# Query encoder (for FAISS dense search)
embed_model = BGEM3FlagModel(
    'BAAI/bge-m3',
    use_fp16=True,
    device='cuda:0'
)
print("✓ Embedding model loaded")

# Reranker (batch_size=16 for A100)
reranker = FlagReranker(
    'BAAI/bge-reranker-v2-m3',
    use_fp16=True,
    device='cuda:0'
)
print("✓ Reranker loaded")

In [ ]:
# ============================================================================
# 3b. (Optional) LLM for query decomposition / routing
# ============================================================================
# By default the notebook runs the rule-based decomposer, which only splits on
# semicolons + a few conjunctions and effectively never decomposes (~4% of the
# 2k set, fragmentary). To enable real facet-aware decomposition, flip
# USE_LLM_DECOMPOSE=True below. This loads the project's built-in HF LLM
# (Qwen2.5-7B-Instruct in 4-bit, ~5 GB) ONCE and adapts its chat-style
# signature to the (prompt: str) -> str the decomposer/router expect.
#
# GPU note: build_hf_llm_call loads on cuda:0 in 4-bit. On an A100 it
# co-resides with the bge-m3 embedder + bge-reranker. If VRAM is tight, set
# USE_LLM_DECOMPOSE=False to keep the original rule-only run, or point this at
# a hosted model (Gemini/OpenAI) with a matching (str)->str callable instead.
#
# The llm_call is shared with the reranker's _gpu_lock? No — generation is a
# separate model; it does NOT take _gpu_lock. The 4 worker threads each call
# decompose() which may invoke the LLM. Transformers generate() is NOT
# thread-safe on one model instance, so we guard it with _llm_lock below.
USE_LLM_DECOMPOSE = True   # <- flip to False to restore the rule-only run

import threading as _t
_llm_lock = _t.Lock()
llm_call = None

if USE_LLM_DECOMPOSE:
    from generation.generator import build_hf_llm_call

    _chat = build_hf_llm_call(
        model_name="Qwen/Qwen2.5-7B-Instruct",
        load_in_4bit=True,
        max_new_tokens=512,     # JSON schema is small; 512 is plenty + safe
        temperature=0.1,
        max_input_tokens=3072,
        gpu_index=0,
    )
    print("✓ Qwen2.5-7B-Instruct (4-bit) loaded for decomposition/routing")

    def llm_call(prompt: str) -> str:
        # generate() is not thread-safe on a single model instance; serialise.
        with _llm_lock:
            return _chat([{"role": "user", "content": prompt}])
else:
    print("(LLM decomposition disabled — using rule-based decomposer/router)")


In [ ]:
# ============================================================================
# 4. Build the retrieval pipeline
# ============================================================================
from retrieval.router import Router, RouterConfig
from retrieval.decomposer import Decomposer, DecomposerConfig
from retrieval.graph_expand import GraphExpander
from retrieval.final_selector import FinalSelector, SelectionConfig, LANE_BUDGETS
from retrieval.retrieval_pipeline import (
    RetrievalPipeline,
    RetrievalConfig,
)
from retrieval.bm25_index import tokenize_query
import numpy as np

# BM25 search callable — THREAD-SAFE: SQLite connections can't cross
# threads, so each worker opens its own read-only connection (cached in
# thread-local storage). The FTS5 index is read-only and shared on disk.
import sqlite3, threading
_tls = threading.local()
def _bm25_conn():
    conn = getattr(_tls, 'conn', None)
    if conn is None:
        conn = sqlite3.connect(f'file:{DATA / "chunk_store.sqlite"}?mode=ro',
                               uri=True, check_same_thread=False)
        _tls.conn = conn
    return conn

def bm25_search(query: str, top_k: int):
    tokens = tokenize_query(query)
    if not tokens:
        return []
    fts_query = ' OR '.join(f'"{t}"' for t in tokens[:20])
    sql = f"""
        SELECT c.row_idx, f.rank, c.law_id, c.ten_van_ban, c.dieu_so
        FROM chunks_fts f
        JOIN chunks c ON c.row_idx = f.rowid
        WHERE chunks_fts MATCH ?
        ORDER BY f.rank
        LIMIT ?
    """
    rows = _bm25_conn().execute(sql, (fts_query, top_k)).fetchall()
    return [
        {
            'row_idx': r[0],
            'score': -float(r[1]),  # rank is negative; negate for descending
            'law_id': r[2],
            'ten_van_ban': r[3],
            'dieu_so': r[4],
        }
        for r in rows
    ]

# GPU locks: one shared GPU -> serialise the actual kernel launches, but
# the calls release the GIL, so 3 other threads run CPU stages (BM25,
# graph expand, RRF, final select) while one holds the GPU. This is the
# single-GPU threading win. encode/search/compute_score are the hot GPU
# calls; guarding them avoids CUDA stream contention on one context.
import threading as _threading
_gpu_lock = _threading.Lock()

# FAISS dense search callable (O(1) metadata lookup via pos_to_meta)
def dense_search(query: str, top_k: int):
    with _gpu_lock:
        emb = embed_model.encode([query], return_dense=True, return_sparse=False)['dense_vecs']
    emb = emb.astype('float32')
    faiss.normalize_L2(emb)
    with _gpu_lock:
        D, I = faiss_gpu.search(emb, top_k)
    hits = []
    for dist, idx in zip(D[0], I[0]):
        if idx < 0:
            continue
        rec = pos_to_meta.get(int(idx))
        if rec is None:
            continue
        hits.append({
            'row_idx': int(rec['row_idx']),
            'score': float(dist),
            'law_id': rec['law_id'],
            'ten_van_ban': rec['ten_van_ban'],
            'dieu_so': rec['dieu_so'],
        })
    return hits

# Text provider (O(1) dict lookup from the prebuilt text cache)
def text_provider(row_idxs):
    return {int(r): text_by_row.get(int(r), '') for r in row_idxs}

# Reranker callable (batch mode). compute_score is itself batched on GPU; we
# hand it ALL pairs at once and let batch_size chunk them — far faster than
# per-pair calls. 64 is comfortable for an A100's 40-80GB with bge-reranker-v2-m3.
def rerank_fn(query: str, passages):
    if not passages:
        return []
    pairs = [[query, p] for p in passages]
    with _gpu_lock:
        scores = reranker.compute_score(pairs, batch_size=64, normalize=True)
    if isinstance(scores, (int, float)):
        return [float(scores)]
    return [float(s) for s in scores]

# Graph expander
graph_expander = GraphExpander(G, row_to_uid)

# Build pipeline
# --- F2 recall-first final selection (agent_instruction/final_select_prompt.md) ---
# The grader scores macro-F2 on Điều-X matches with recall weighted 4x over
# precision. The default LANE_BUDGETS collapse to top-1 for direct_lookup and
# stay tight elsewhere, which drops the gold article whenever the top rerank
# hit is semantically close but legally irrelevant (data/results-5.json id
# 127/128: 'thanh tra viên xử phạt thuế' returned archival + fire decrees).
# For F2, recall is paramount and precision is cheap, so we widen the per-lane
# article budgets, loosen the admission margin, and raise the recall floor.
# Article-first grouping + redundancy penalty (unchanged) keep noise down.
#
# NO reliable local ground truth exists (dev_set/ground_truth.json is noise);
# the HOST returns the real F2 on submission. So the knobs below are exposed
# as variables: change them, re-run the batch, submit, read the host score,
# iterate. Sweep REL_MARGIN and the per-lane MAX_ARTICLES_* first.
import retrieval.final_selector as _fs

# --- Tunable knobs (sweep these, submit, read host F2) ---
REL_MARGIN          = 0.75   # was 0.45: admit runner-up within 75% of top (recall)
MIN_ARTICLES        = 2      # was 1: never collapse to a single article
REDUNDANCY_PENALTY  = 0.03   # was 0.04: softer — keep legal-chain siblings
MAX_ARTICLES_DIRECT = 5      # was 1: keep runners-up; gold often 2nd-3rd
MAX_ARTICLES_PROC   = 5      # was 2: governing + detailing + support
MAX_ARTICLES_COND   = 5      # was 3: resolve ALL conditions -> more articles
MAX_ARTICLES_SANCT  = 4      # was 2: effective + replacing + sanction chain
MAX_ARTICLES_CROSS  = 8      # was 4: keep the full multi-hop legal chain
MAX_ARTICLES_SCEN   = 8      # was 6: already wide; keep
MAX_CHUNKS_PER_ART  = 4      # unchanged

_fs.LANE_BUDGETS.update({
    'direct_lookup':         (MAX_ARTICLES_DIRECT, MAX_CHUNKS_PER_ART),
    'procedure_detail':      (MAX_ARTICLES_PROC,   MAX_CHUNKS_PER_ART),
    'condition_requirement': (MAX_ARTICLES_COND,   MAX_CHUNKS_PER_ART),
    'sanction':              (MAX_ARTICLES_SANCT,  MAX_CHUNKS_PER_ART),
    'cross_doc':             (MAX_ARTICLES_CROSS,  MAX_CHUNKS_PER_ART),
    'scenario':              (MAX_ARTICLES_SCEN,   MAX_CHUNKS_PER_ART),
})

router = Router(
    llm_call=llm_call,
    config=RouterConfig(use_llm=(llm_call is not None)),
)
decomposer = Decomposer(
    llm_call=llm_call,
    config=DecomposerConfig(use_llm=(llm_call is not None)),
)
selector = FinalSelector(config=SelectionConfig(
    drop_provincial=False,        # keep provincial văn bản (grader grades on them)
    rel_margin=REL_MARGIN,
    min_articles=MIN_ARTICLES,
    redundancy_penalty=REDUNDANCY_PENALTY,
))

pipeline = RetrievalPipeline(
    router=router,
    decomposer=decomposer,
    lexical_search=bm25_search,
    dense_search=dense_search,
    reranker=rerank_fn,
    text_provider=text_provider,
    graph_expander=graph_expander,
    final_selector=selector,
    cfg=RetrievalConfig(
        top_bm25=80,
        top_dense=80,
        rrf_k=60,
        fused_pool_size=120,
        use_dense=True,
        rerank_input_size=70,
        rerank_text_truncate=512,
        graph_expand_seeds=8,
        graph_expanded_top=60,
    ),
)

print("✓ Pipeline ready")


# ---- Final-select prompt-JSON renderer -------------------------------------
# Mirrors the output contract in agent_instruction/final_select_prompt.md:
#   { selected_articles:[{article_id,score,reason,supporting_chunks:[{chunk_id,
#      score,reason}]}], notes }
# Principle: select articles first, chunks second; prefer minimal sufficient
# legal context over raw similarity. This does NOT replace the grader record
# (relevant_docs / relevant_articles) — it makes the selection stage itself
# observable in the agent-prompt's required JSON shape.
def final_select_prompt_json(result):
    sel = getattr(result, "selection", None)
    if sel is None:
        return {"selected_articles": [], "notes": "no selection produced"}
    lane = getattr(result.routing, "lane", "direct_lookup")
    meta = getattr(sel, "selection_metadata", {}) or {}
    budget = meta.get("budget") or {}

    def _art_reason(a):
        parts = []
        if a.relevance:
            parts.append(f"relevance {a.relevance:.3f}")
        if a.support:
            parts.append(f"support +{a.support:.3f}")
        if a.graph_gain:
            parts.append(f"graph_gain +{a.graph_gain:.3f}")
        if a.authority:
            parts.append(f"authority +{a.authority:.3f}")
        if a.redundancy:
            parts.append(f"redundancy -{a.redundancy:.3f}")
        return "; ".join(parts) or "selected"

    def _chunk_reason(c):
        return ("graph-expanded" if c.from_graph else "rerank seed") + f", score {c.score:.3f}"

    arts = []
    for a in getattr(sel, "articles", []):
        arts.append({
            "article_id": a.to_relevant_string(),
            "score": round(float(a.final_score), 4),
            "reason": _art_reason(a),
            "supporting_chunks": [
                {
                    "chunk_id": str(c.chunk_id) if c.chunk_id is not None else f"row:{c.row_idx}",
                    "score": round(float(c.score), 4),
                    "reason": _chunk_reason(c),
                }
                for c in getattr(a, "chunks", [])
            ],
        })

    notes = (
        f"lane={lane}; article-first selection "
        f"(budget articles={budget.get('max_articles')}, chunks/art={budget.get('max_chunks')}); "
        f"kept {len(arts)} article(s) of {meta.get('n_candidates', '?')} candidates; "
        f"minimal sufficient legal chain, redundancy-penalised."
    )
    return {"selected_articles": arts, "notes": notes}


In [ ]:
# ============================================================================
# 3c. Recall-first enhancer (ported from kaggle-hybridrag-decomp-anchor-v2)
# ============================================================================
# Layers the anchor notebook's recall-first retrieval on top of the modular
# RetrievalPipeline. REUSES the pipeline's callables (lexical_search,
# dense_search, reranker, text_provider) + the llm_call from cell 3b — no
# models are loaded twice.
#
# Flow: LLM query plan -> multi-variant retrieval (original + atomic + facet +
#       must-terms) -> per-variant rerank -> article aggregation (support
#       bonuses + domain priors + facet scores) -> facet-coverage selection
#       (complexity-adaptive budgets + family caps + preferred-doc rescue)
#
# retrieve_recall_first(query) returns an object with relevant_docs() and
# relevant_articles() methods (compatible with RetrievalResult), so it is a
# drop-in for pipeline.retrieve() in the smoke test and batch cells. On any
# exception it falls back to pipeline.retrieve(query) so the batch never dies.
#
# Toggle with USE_RECALL_FIRST below. Set to False to use the original modular
# pipeline (faster but lower recall).

USE_RECALL_FIRST = True

import re
import math
import unicodedata
from collections import defaultdict, OrderedDict
from retrieval.article_select import canonical_dieu
from retrieval.final_selector import ChunkEvidence, ScoredArticle, SelectionResult

# --- Config knobs (sweep these, submit, read host F2) ---
RC_MAX_ATOMIC          = 4      # max atomic sub-questions per query
RC_MAX_FACETS          = 6      # max facet variants per query
RC_USE_MUST_TERMS      = True   # add a must-terms lexical-only variant
RC_BM25_TOPK           = 40     # per-variant BM25 depth
RC_DENSE_TOPK          = 100    # per-variant dense depth
RC_RRF_K               = 60     # RRF smoothing constant
RC_CANDIDATE_TOPK      = 96     # per-variant fused pool size
RC_RERANK_TOPK         = 48     # per-variant rerank depth
RC_RERANK_TRUNCATE     = 512    # chars per passage handed to reranker
RC_ARTICLE_MAX_SIMPLE  = 2
RC_ARTICLE_MAX_MEDIUM  = 4
RC_ARTICLE_MAX_COMPLEX = 12

# --- Text utilities (ported from anchor) ---
_rc_word_re = re.compile(r'\w+', re.UNICODE)
RC_STOPWORDS = {
    'và', 'hoặc', 'của', 'các', 'những', 'một', 'này', 'đó', 'thì', 'là',
    'có', 'bị', 'được', 'phải', 'cho', 'về', 'trong', 'ngoài', 'theo', 'nếu',
    'khi', 'như', 'để', 'với', 'từ', 'ra', 'sao', 'gì', 'nào', 'bao', 'nhiêu',
    'trường', 'hợp', 'cần', 'muốn', 'hỏi', 'tôi', 'công', 'ty',
}


def rc_normalize_text(text):
    text = str(text).replace('\u200b', ' ').replace('\ufeff', ' ')
    return re.sub(r'\s+', ' ', text).strip()


def rc_normalize_key(text):
    return rc_normalize_text(text).lower()


def rc_tokenize_lexical(text, keep_stopwords=False):
    toks = _rc_word_re.findall(rc_normalize_text(text).lower())
    toks = [t for t in toks if len(t) >= 2]
    if not keep_stopwords:
        toks = [t for t in toks if t not in RC_STOPWORDS]
    return toks


def rc_anchor_plain(text):
    # Strip diacritics + lower + replace đ -> d, for fuzzy domain matching.
    text = unicodedata.normalize('NFD', rc_normalize_text(text).lower())
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.replace('\u0111', 'd')


def _rc_plain_has_any(plain, needles):
    return any(needle in plain for needle in needles)


def _rc_append_unique(values, value):
    value = rc_normalize_text(value)
    if value and value not in values:
        values.append(value)


# --- Dedup / similarity helpers ---
def rc_lexical_jaccard(a, b):
    aa = set(rc_tokenize_lexical(a, keep_stopwords=False))
    bb = set(rc_tokenize_lexical(b, keep_stopwords=False))
    if not aa or not bb:
        return 0.0
    return len(aa & bb) / max(len(aa | bb), 1)


def rc_atomic_questions_too_similar(items):
    if len(items) < 2:
        return False
    pairs = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            pairs.append(rc_lexical_jaccard(items[i], items[j]))
    return bool(pairs) and max(pairs) >= 0.62


def rc_dedupe_atomic_questions(items, max_items=None):
    out = []
    for item in items:
        item = rc_normalize_text(item)
        if not item:
            continue
        if any(rc_normalize_key(item) == rc_normalize_key(x) or
               rc_lexical_jaccard(item, x) >= 0.78 for x in out):
            continue
        out.append(item)
        if max_items and len(out) >= max_items:
            break
    return out


def rc_sanitize_list_of_strings(values, max_items=8, max_chars=220):
    out, seen = [], set()
    if not isinstance(values, list):
        return out
    for value in values:
        text = rc_normalize_text(str(value))[:max_chars].strip(' -')
        key = rc_normalize_key(text)
        if text and key not in seen:
            out.append(text)
            seen.add(key)
        if len(out) >= max_items:
            break
    return out


# --- Generic atomic question detection ---
def rc_is_generic_atomic_question(text, domain_profile=None):
    plain = rc_anchor_plain(text)
    generic_markers = [
        'tai lieu', 'chung cu', 'ho so', 'don yeu cau',
        'hop dong', 'tranh chap', 'xu ly',
    ]
    if not _rc_plain_has_any(plain, generic_markers):
        return False
    domain_profile = domain_profile or {}
    anchors = domain_profile.get('anchor_terms', [])
    return not any(rc_anchor_plain(anchor) in plain for anchor in anchors)


def rc_repair_generic_atomic_question(text, domain_profile):
    text = rc_normalize_text(text)
    if not rc_is_generic_atomic_question(text, domain_profile):
        return text
    anchors = [a for a in domain_profile.get('anchor_terms', []) if rc_normalize_text(a)]
    if not anchors:
        return text
    suffix = ' về ' + ', '.join(anchors[:3])
    if rc_anchor_plain(suffix) in rc_anchor_plain(text):
        return text
    if text.endswith('?'):
        return text[:-1].rstrip() + suffix + '?'
    return text + suffix


# --- Domain profile (hard-coded legal domain priors) ---
def rc_build_rule_domain_profile(question, must_terms=None, raw_anchor_terms=None):
    # Detect legal domain from the question text and return preferred law IDs,
    # preferred/negative title terms, and anchor terms. These priors boost
    # domain-correct articles and suppress domain-drift articles in scoring.
    must_terms = must_terms or []
    raw_anchor_terms = raw_anchor_terms or []
    haystack = ' '.join([question] + list(must_terms) + list(raw_anchor_terms))
    plain = rc_anchor_plain(haystack)
    labels, anchors, preferred_law_ids = [], [], []
    preferred_title_terms, negative_title_terms, soft_negative_law_ids = [], [], []

    def add_label(label):
        if label not in labels:
            labels.append(label)

    def add_anchor(text):
        _rc_append_unique(anchors, text)

    def add_pref_law(law_id):
        if law_id not in preferred_law_ids:
            preferred_law_ids.append(law_id)

    def add_pref_title(text):
        _rc_append_unique(preferred_title_terms, text)

    def add_negative_title(text):
        _rc_append_unique(negative_title_terms, text)

    def add_soft_negative_law(law_id):
        if law_id not in soft_negative_law_ids:
            soft_negative_law_ids.append(law_id)

    # --- Copyright / software ---
    has_copyright = _rc_plain_has_any(plain, [
        'quyen tac gia', 'quyen lien quan', 'phan mem',
        'chuong trinh may tinh', 'sao chep', 'xam pham quyen tac gia',
    ])
    has_industrial = _rc_plain_has_any(plain, [
        'so huu cong nghiep', 'nhan hieu', 'sang che', 'kieu dang cong nghiep',
    ])
    if has_copyright:
        add_label('copyright_software')
        for term in [
            'quyền tác giả', 'phần mềm', 'sao chép', 'cho thuê',
            'chương trình máy tính', 'xâm phạm quyền tác giả',
        ]:
            if rc_anchor_plain(term) in plain:
                add_anchor(term)
        add_pref_law('50/2005/QH11')
        add_pref_law('17/2023/NĐ-CP')
        for term in ['quyền tác giả', 'quyền liên quan', 'sở hữu trí tuệ']:
            add_pref_title(term)
        if not has_industrial:
            add_negative_title('sở hữu công nghiệp')
            add_negative_title('giống cây trồng')
            add_soft_negative_law('65/2023/NĐ-CP')
            add_soft_negative_law('99/2013/NĐ-CP')
            add_soft_negative_law('11/2015/TT-BKHCN')

    # --- Customs ---
    if _rc_plain_has_any(plain, ['hai quan', 'nhap khau', 'kiem soat']):
        add_label('customs_control')
        add_anchor('hải quan')
        add_anchor('kiểm soát')
        add_anchor('nhập khẩu')
        add_pref_law('13/2015/TT-BTC')
        add_pref_law('50/2005/QH11')
        add_pref_title('hải quan')
        add_pref_title('kiểm soát')

    # --- Copyright assessment ---
    if _rc_plain_has_any(plain, ['giam dinh', 'giam dinh vien']):
        add_label('copyright_assessment')
        add_anchor('giám định')
        add_pref_law('15/2012/TT-BVHTTDL')
        add_pref_law('17/2023/NĐ-CP')
        add_pref_law('105/2006/NĐ-CP')
        add_pref_title('giám định quyền tác giả')
        add_negative_title('chuyển giao công nghệ')
        add_negative_title('thương mại')

    # --- Consumer protection ---
    if _rc_plain_has_any(plain, ['nguoi tieu dung', 'de bi ton thuong']):
        add_label('consumer_protection')
        add_anchor('người tiêu dùng')
        add_anchor('dễ bị tổn thương')
        add_pref_law('19/2023/QH15')
        add_pref_title('người tiêu dùng')

    # --- Small business + procurement ---
    if (_rc_plain_has_any(plain, ['doanh nghiep nho va vua', 'dnnvv']) and
            _rc_plain_has_any(plain, ['dau thau', 'lua chon nha thau'])):
        add_label('small_business_procurement')
        add_anchor('doanh nghiệp nhỏ và vừa')
        add_anchor('đấu thầu')
        add_pref_law('04/2017/QH14')
        add_pref_law('22/2023/QH15')
        add_pref_title('hỗ trợ doanh nghiệp nhỏ và vừa')
        add_pref_title('đấu thầu')

    for term in raw_anchor_terms:
        add_anchor(term)

    return {
        'labels': labels,
        'anchor_terms': anchors[:12],
        'preferred_law_ids': preferred_law_ids[:12],
        'preferred_title_terms': preferred_title_terms[:12],
        'negative_title_terms': negative_title_terms[:12],
        'soft_negative_law_ids': soft_negative_law_ids[:12],
    }


# --- Facet profiles ---
def rc_facet_profile(facet_id, label, query, anchor_terms=None,
                     preferred_law_ids=None, preferred_title_terms=None,
                     target_terms=None, negative_terms=None, priority=1.0):
    return {
        'facet_id': str(facet_id),
        'label': rc_normalize_text(label),
        'query': rc_normalize_text(query),
        'anchor_terms': [rc_normalize_text(x) for x in (anchor_terms or []) if rc_normalize_text(x)],
        'preferred_law_ids': [str(x).strip() for x in (preferred_law_ids or []) if str(x).strip()],
        'preferred_title_terms': [rc_normalize_text(x) for x in (preferred_title_terms or []) if rc_normalize_text(x)],
        'target_terms': [rc_normalize_text(x) for x in (target_terms or []) if rc_normalize_text(x)],
        'negative_terms': [rc_normalize_text(x) for x in (negative_terms or []) if rc_normalize_text(x)],
        'priority': float(priority),
    }


def rc_sanitize_facet_profiles(values, max_items=None):
    max_items = max_items or max(6, RC_MAX_ATOMIC)
    out, seen = [], set()
    if not isinstance(values, list):
        return out
    for raw in values:
        if not isinstance(raw, dict):
            continue
        facet_id = rc_normalize_key(str(raw.get('facet_id') or raw.get('label') or raw.get('query') or 'facet'))
        facet_id = re.sub(r'[^a-z0-9_]+', '_', facet_id).strip('_')[:64] or 'facet'
        query = rc_normalize_text(raw.get('query') or raw.get('text') or raw.get('label') or '')
        label = rc_normalize_text(raw.get('label') or query or facet_id)
        key = facet_id + '|' + rc_normalize_key(query or label)
        if not query or key in seen:
            continue
        out.append(rc_facet_profile(
            facet_id=facet_id, label=label, query=query,
            anchor_terms=rc_sanitize_list_of_strings(raw.get('anchor_terms', []), max_items=8, max_chars=80),
            preferred_law_ids=rc_sanitize_list_of_strings(raw.get('preferred_law_ids', []), max_items=8, max_chars=40),
            preferred_title_terms=rc_sanitize_list_of_strings(raw.get('preferred_title_terms', []), max_items=8, max_chars=80),
            target_terms=rc_sanitize_list_of_strings(raw.get('target_terms', []), max_items=12, max_chars=80),
            negative_terms=rc_sanitize_list_of_strings(raw.get('negative_terms', []), max_items=8, max_chars=80),
            priority=float(raw.get('priority', 1.0) or 1.0),
        ))
        seen.add(key)
        if len(out) >= max_items:
            break
    return out


def rc_build_rule_facet_profiles(question, domain_profile):
    # Generate facet profiles (one per legal aspect) from domain labels.
    # Each facet carries preferred_law_ids + target_terms so the facet-coverage
    # selection can match articles to the right legal aspect.
    plain = rc_anchor_plain(question)
    labels = set(domain_profile.get('labels', []))
    facets = []

    def add(facet):
        key = facet['facet_id']
        if key not in {x['facet_id'] for x in facets}:
            facets.append(facet)

    if 'small_business_procurement' in labels:
        add(rc_facet_profile(
            'small_business_support',
            'Ưu đãi hỗ trợ doanh nghiệp nhỏ và vừa',
            'Ưu đãi, hỗ trợ dành cho doanh nghiệp nhỏ và vừa theo pháp luật hỗ trợ doanh nghiệp nhỏ và vừa?',
            anchor_terms=['doanh nghiệp nhỏ và vừa', 'ưu đãi', 'hỗ trợ'],
            preferred_law_ids=['04/2017/QH14'],
            preferred_title_terms=['hỗ trợ doanh nghiệp nhỏ và vừa'],
            target_terms=['doanh nghiệp nhỏ và vừa', 'hỗ trợ', 'ưu đãi', 'chính sách hỗ trợ'],
            negative_terms=['chuyển giao công nghệ', 'thương mại'],
            priority=1.15,
        ))
        add(rc_facet_profile(
            'procurement_preference',
            'Ưu đãi trong đấu thầu',
            'Ưu đãi đối với doanh nghiệp nhỏ và vừa trong lựa chọn nhà thầu, đấu thầu?',
            anchor_terms=['doanh nghiệp nhỏ và vừa', 'đấu thầu', 'lựa chọn nhà thầu'],
            preferred_law_ids=['22/2023/QH15'],
            preferred_title_terms=['đấu thầu'],
            target_terms=['ưu đãi', 'đấu thầu', 'nhà thầu', 'lựa chọn nhà thầu', 'doanh nghiệp nhỏ và vừa'],
            negative_terms=['chuyển giao công nghệ', 'thương mại'],
            priority=1.14,
        ))

    if 'copyright_software' in labels:
        multi_domain = bool(labels & {'customs_control', 'copyright_assessment', 'consumer_protection'})
        explicit_software = _rc_plain_has_any(plain, ['phan mem', 'chuong trinh may tinh', 'sao chep', 'cho thue'])
        if (not multi_domain) or explicit_software:
            add(rc_facet_profile(
                'software_property_rights',
                'Quyền tài sản với chương trình máy tính',
                'Quyền tác giả và quyền tài sản đối với chương trình máy tính, phần mềm được quy định thế nào?',
                anchor_terms=['quyền tác giả', 'phần mềm', 'chương trình máy tính'],
                preferred_law_ids=['50/2005/QH11'],
                preferred_title_terms=['sở hữu trí tuệ', 'quyền tác giả'],
                target_terms=['quyền tài sản', 'chương trình máy tính', 'phần mềm', 'sao chép', 'cho thuê'],
                negative_terms=['sở hữu công nghiệp', 'nhãn hiệu', 'sáng chế'],
                priority=1.12,
            ))
        if _rc_plain_has_any(plain, ['sao chep', 'cho thue', 'xam pham']):
            add(rc_facet_profile(
                'copyright_infringement',
                'Hành vi xâm phạm quyền tác giả phần mềm',
                'Hành vi sao chép, cho thuê trái phép phần mềm xâm phạm quyền tác giả như thế nào?',
                anchor_terms=['quyền tác giả', 'phần mềm', 'sao chép', 'cho thuê'],
                preferred_law_ids=['50/2005/QH11'],
                preferred_title_terms=['sở hữu trí tuệ', 'quyền tác giả'],
                target_terms=['xâm phạm', 'sao chép', 'cho thuê', 'quyền tác giả', 'chương trình máy tính'],
                negative_terms=['sở hữu công nghiệp', 'nhãn hiệu', 'sáng chế'],
                priority=1.11,
            ))
        if _rc_plain_has_any(plain, ['ton that', 'mat khach hang', 'co hoi kinh doanh', 'thiet hai', 'boi thuong']):
            add(rc_facet_profile(
                'damage_business_opportunity',
                'Thiệt hại và cơ hội kinh doanh',
                'Cách xác định thiệt hại và tổn thất cơ hội kinh doanh do xâm phạm quyền tác giả phần mềm?',
                anchor_terms=['quyền tác giả', 'phần mềm', 'thiệt hại', 'cơ hội kinh doanh'],
                preferred_law_ids=['50/2005/QH11', '17/2023/NĐ-CP'],
                preferred_title_terms=['sở hữu trí tuệ', 'quyền tác giả'],
                target_terms=['thiệt hại', 'tổn thất', 'cơ hội kinh doanh', 'bồi thường', 'mất khách hàng'],
                negative_terms=['sở hữu công nghiệp', 'nhãn hiệu', 'sáng chế'],
                priority=1.10,
            ))
        if _rc_plain_has_any(plain, ['tu bao ve', 'yeu cau xu ly', 'bien phap bao ve', 'xu ly xam pham']):
            add(rc_facet_profile(
                'self_protection_request',
                'Quyền tự bảo vệ và yêu cầu xử lý',
                'Quyền tự bảo vệ, yêu cầu xử lý xâm phạm và biện pháp bảo vệ quyền sở hữu trí tuệ được quy định thế nào?',
                anchor_terms=['quyền tác giả', 'xâm phạm', 'yêu cầu xử lý'],
                preferred_law_ids=['50/2005/QH11'],
                preferred_title_terms=['sở hữu trí tuệ'],
                target_terms=['quyền tự bảo vệ', 'yêu cầu xử lý', 'biện pháp bảo vệ', 'xâm phạm quyền sở hữu trí tuệ'],
                negative_terms=['sở hữu công nghiệp', 'nhãn hiệu', 'sáng chế'],
                priority=1.08,
            ))
        if _rc_plain_has_any(plain, ['tai lieu', 'chung cu', 'don yeu cau', 'xu ly']):
            add(rc_facet_profile(
                'evidence_request_docs',
                'Tài liệu chứng cứ yêu cầu xử lý',
                'Tài liệu, chứng cứ cần chuẩn bị khi yêu cầu xử lý hành vi xâm phạm quyền tác giả đối với phần mềm?',
                anchor_terms=['quyền tác giả', 'phần mềm', 'tài liệu', 'chứng cứ'],
                preferred_law_ids=['17/2023/NĐ-CP', '50/2005/QH11'],
                preferred_title_terms=['quyền tác giả', 'sở hữu trí tuệ'],
                target_terms=['tài liệu', 'chứng cứ', 'đơn yêu cầu', 'yêu cầu xử lý', 'xâm phạm quyền tác giả'],
                negative_terms=['sở hữu công nghiệp', 'nhãn hiệu', 'sáng chế'],
                priority=1.06,
            ))

    if 'customs_control' in labels:
        add(rc_facet_profile(
            'customs_financial_guarantee',
            'Kiểm soát hải quan và bảo đảm tài chính',
            'Nghĩa vụ bảo đảm tài chính khi yêu cầu hải quan kiểm soát hàng hóa nghi xâm phạm quyền tác giả?',
            anchor_terms=['hải quan', 'kiểm soát', 'nhập khẩu', 'bảo đảm tài chính'],
            preferred_law_ids=['13/2015/TT-BTC', '50/2005/QH11'],
            preferred_title_terms=['hải quan', 'sở hữu trí tuệ'],
            target_terms=['hải quan', 'kiểm soát', 'hàng hóa', 'bảo đảm tài chính', 'tạm dừng làm thủ tục'],
            negative_terms=['chuyển giao công nghệ', 'thương mại'],
            priority=1.16,
        ))
    if 'copyright_assessment' in labels:
        add(rc_facet_profile(
            'assessment_contract',
            'Hợp đồng giám định quyền tác giả',
            'Hợp đồng giám định quyền tác giả, quyền liên quan cần có những nội dung chính nào?',
            anchor_terms=['giám định', 'hợp đồng giám định', 'quyền tác giả'],
            preferred_law_ids=['15/2012/TT-BVHTTDL', '17/2023/NĐ-CP', '105/2006/NĐ-CP'],
            preferred_title_terms=['giám định quyền tác giả', 'quyền tác giả'],
            target_terms=['giám định', 'hợp đồng giám định', 'giám định viên', 'nội dung hợp đồng'],
            negative_terms=['chuyển giao công nghệ', 'thương mại'],
            priority=1.15,
        ))
    if 'consumer_protection' in labels:
        add(rc_facet_profile(
            'consumer_dispute',
            'Tranh chấp với người tiêu dùng dễ bị tổn thương',
            'Trách nhiệm giải quyết tranh chấp khi đối tượng bị xâm phạm là người tiêu dùng dễ bị tổn thương?',
            anchor_terms=['người tiêu dùng', 'dễ bị tổn thương', 'tranh chấp'],
            preferred_law_ids=['19/2023/QH15'],
            preferred_title_terms=['người tiêu dùng'],
            target_terms=['người tiêu dùng', 'dễ bị tổn thương', 'tranh chấp', 'trách nhiệm'],
            negative_terms=['chuyển giao công nghệ', 'thương mại'],
            priority=1.14,
        ))

    return facets[:max(6, RC_MAX_FACETS)]


def rc_merge_facet_profiles(rule_facets, raw_facets):
    out, seen = [], set()
    for facet in list(rule_facets or []) + list(raw_facets or []):
        if not isinstance(facet, dict):
            continue
        key = facet.get('facet_id', '') + '|' + rc_normalize_key(facet.get('query', ''))
        if not key.strip('|') or key in seen:
            continue
        out.append(facet)
        seen.add(key)
    return out[:max(6, RC_MAX_FACETS)]


# --- Query plan (LLM + rule fallback) ---
COMPLEXITIES = {'simple', 'medium', 'complex'}
QUESTION_TYPES = {
    'procedure', 'condition', 'rights_obligations', 'sanction',
    'support_incentive', 'scenario', 'deadline', 'comparison',
    'definition_listing', 'other',
}


def rc_detect_question_type(question):
    q = rc_normalize_key(question)
    if re.search(r'\b(quy định|liệt kê|bao gồm|gồm|những|các)\b.*\b(nào|gì)\b', q) and not re.search(r'\b(thủ tục|hồ sơ|điều kiện|xử phạt|vi phạm|trách nhiệm|nghĩa vụ|ưu đãi|hỗ trợ)\b', q):
        return 'definition_listing'
    if re.search(r'\b(thủ tục|hồ sơ|tài liệu|chứng cứ|đơn yêu cầu|chuẩn bị)\b', q):
        return 'procedure'
    if re.search(r'\b(điều kiện|yêu cầu|tiêu chí|đáp ứng)\b', q):
        return 'condition'
    if re.search(r'\b(quyền|nghĩa vụ|trách nhiệm)\b', q):
        return 'rights_obligations'
    if re.search(r'\b(phạt|xử phạt|vi phạm|xử lý|khắc phục)\b', q):
        return 'sanction'
    if re.search(r'\b(hỗ trợ|ưu đãi|miễn|giảm)\b', q):
        return 'support_incentive'
    if re.search(r'\b(thời hạn|bao lâu|khi nào|mấy ngày)\b', q):
        return 'deadline'
    if re.search(r'\b(khác gì|khác biệt|so sánh)\b', q):
        return 'comparison'
    if len(rc_tokenize_lexical(question, keep_stopwords=True)) >= 45:
        return 'scenario'
    return 'other'


def rc_is_simple_listing_query(question):
    q = rc_normalize_key(question)
    toks = rc_tokenize_lexical(question, keep_stopwords=True)
    if len(toks) > 16:
        return False
    if re.search(r'\b(thủ tục|hồ sơ|điều kiện|xử phạt|vi phạm|khắc phục|trách nhiệm|nghĩa vụ|ưu đãi|hỗ trợ|đấu thầu|bồi thường)\b', q):
        return False
    listing_patterns = [
        r'\bquy định\s+(?:những|các)?\s*.+\s+nào\b',
        r'\b(?:những|các)\s+.+\s+nào\b',
        r'\b.+\s+bao gồm\s+(?:những|các)?\s+gì\b',
    ]
    return any(re.search(pat, q) for pat in listing_patterns)


def rc_split_question_clauses(question):
    q = rc_normalize_text(question)
    pieces = re.split(
        r'[;?。]+|\s+(?:đồng thời|ngoài ra|bên cạnh đó|trong trường hợp|nếu|khi|và nếu|và phải|và cần|đặc biệt nếu)\s+',
        q, flags=re.IGNORECASE,
    )
    out = []
    for piece in pieces:
        piece = rc_normalize_text(piece.strip(' ,.-:'))
        if 25 <= len(piece) <= 300:
            out.append(piece)
    if len(out) <= 1 and len(q) > 120:
        for piece in re.split(r',\s+| và ', q):
            piece = rc_normalize_text(piece.strip(' ,.-:'))
            if 25 <= len(piece) <= 260:
                out.append(piece)
    seen, deduped = set(), []
    for piece in out:
        key = rc_normalize_key(piece)
        if key not in seen and key != rc_normalize_key(q):
            seen.add(key)
            deduped.append(piece)
    return deduped[:RC_MAX_ATOMIC]


def rc_fallback_complexity(question, clauses=None):
    toks = rc_tokenize_lexical(question, keep_stopwords=True)
    q = rc_normalize_key(question)
    clauses = clauses if clauses is not None else rc_split_question_clauses(question)
    multi_markers = len(re.findall(
        r'\b(và|đồng thời|ngoài ra|nếu|khi|hồ sơ|chứng cứ|xử lý|khắc phục|nghĩa vụ|trách nhiệm|thiệt hại|giám định)\b', q))
    domain_markers = 0
    for pat in [
        r'doanh nghiệp nhỏ và vừa|dnnvv',
        r'đấu thầu|lựa chọn nhà thầu',
        r'quyền tác giả|sở hữu trí tuệ|phần mềm|sao chép',
        r'hải quan|nhập khẩu|xuất khẩu',
        r'giám định',
        r'người tiêu dùng|dễ bị tổn thương',
        r'thuế|đất đai|mặt bằng',
        r'lao động|hợp đồng lao động|bằng cấp|chứng chỉ',
    ]:
        if re.search(pat, q, flags=re.IGNORECASE):
            domain_markers += 1
    if len(toks) >= 50 or len(clauses) >= 3 or multi_markers >= 5:
        return 'complex'
    if domain_markers >= 3:
        return 'complex'
    if len(toks) >= 20 or len(clauses) >= 2 or multi_markers >= 2 or domain_markers >= 2:
        return 'medium'
    return 'simple'


def rc_fallback_must_terms(question, max_terms=10):
    toks = rc_tokenize_lexical(question, keep_stopwords=False)
    seen = []
    for tok in toks:
        if tok not in seen:
            seen.append(tok)
    return seen[:max_terms]


def rc_fallback_query_plan(question, reason='rule_fallback'):
    clauses = rc_split_question_clauses(question)
    complexity = rc_fallback_complexity(question, clauses)
    atomic = clauses if complexity != 'simple' else []
    if not atomic and complexity in {'medium', 'complex'}:
        atomic = [rc_normalize_text(question)]
    plan = {
        'complexity': complexity,
        'atomic_questions': atomic[:RC_MAX_ATOMIC],
        'must_have_terms': rc_fallback_must_terms(question),
        'question_type': rc_detect_question_type(question),
        'rationale_short': reason,
        'planner_fallback': True,
        'planner_error': reason,
        'raw_plan_text': '',
        'anchor_terms': [],
        'legal_facets': [],
        'facet_profiles': [],
        'domain_profile': {},
    }
    return rc_enrich_query_plan(question, rc_refine_query_plan_heuristics(question, plan))


def rc_refine_query_plan_heuristics(question, plan):
    plan = dict(plan or {})
    if rc_is_simple_listing_query(question):
        plan['complexity'] = 'simple'
        plan['atomic_questions'] = []
        plan['question_type'] = 'definition_listing'
        note = 'force_simple_listing'
        existing = rc_normalize_text(plan.get('rationale_short', ''))
        plan['rationale_short'] = (existing + '; ' + note).strip('; ')[:260]
    return plan


def rc_enrich_query_plan(question, plan):
    plan = dict(plan or {})
    must_terms = rc_sanitize_list_of_strings(plan.get('must_have_terms', []), max_items=12, max_chars=80)
    raw_anchors = rc_sanitize_list_of_strings(plan.get('anchor_terms', []), max_items=12, max_chars=80)
    raw_facets = rc_sanitize_facet_profiles(plan.get('facet_profiles', []), max_items=max(6, RC_MAX_FACETS))
    domain_profile = rc_build_rule_domain_profile(question, must_terms=must_terms, raw_anchor_terms=raw_anchors)
    complexity = plan.get('complexity', 'medium')
    rule_facet_profiles = rc_build_rule_facet_profiles(question, domain_profile)
    facet_profiles = rc_merge_facet_profiles(rule_facet_profiles, raw_facets)
    rule_facets = rc_dedupe_atomic_questions(
        [f.get('query', '') for f in facet_profiles] +
        rc_sanitize_list_of_strings(plan.get('legal_facets', []), max_items=6, max_chars=220),
        max_items=max(6, RC_MAX_FACETS),
    )
    atomic = rc_sanitize_list_of_strings(plan.get('atomic_questions', []), max_items=RC_MAX_ATOMIC, max_chars=240)
    if complexity == 'simple':
        atomic = []
    else:
        atomic = [rc_repair_generic_atomic_question(item, domain_profile) for item in atomic]
        labels = set(domain_profile.get('labels', []))
        force_rule_first = (
            complexity == 'complex'
            or 'small_business_procurement' in labels
            or rc_atomic_questions_too_similar(atomic)
            or len(rule_facet_profiles) >= 2
        )
        if rule_facets and force_rule_first:
            atomic = rc_dedupe_atomic_questions(rule_facets + atomic, max_items=RC_MAX_ATOMIC)
        elif rule_facets and len(atomic) < min(2, RC_MAX_ATOMIC):
            atomic = rc_dedupe_atomic_questions(atomic + rule_facets, max_items=RC_MAX_ATOMIC)
        else:
            atomic = rc_dedupe_atomic_questions(atomic, max_items=RC_MAX_ATOMIC)
        if not atomic and complexity in {'medium', 'complex'}:
            atomic = rc_dedupe_atomic_questions(rule_facets or [rc_normalize_text(question)], max_items=RC_MAX_ATOMIC)

    plan['atomic_questions'] = atomic[:RC_MAX_ATOMIC]
    plan['must_have_terms'] = must_terms or rc_fallback_must_terms(question)
    plan['anchor_terms'] = domain_profile.get('anchor_terms', [])
    plan['legal_facets'] = rule_facets
    plan['facet_profiles'] = facet_profiles
    plan['domain_profile'] = domain_profile
    plan['generic_atomic_questions'] = [a for a in atomic if rc_is_generic_atomic_question(a, domain_profile)]
    return plan


def rc_extract_json_object(text):
    text = str(text).strip()
    start = text.find('{')
    end = text.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('No JSON object found in planner output')
    return json.loads(text[start:end + 1])


def rc_token_overlap_ratio(candidate, original):
    cand = set(rc_tokenize_lexical(candidate, keep_stopwords=False))
    orig = set(rc_tokenize_lexical(original, keep_stopwords=False))
    if not cand or not orig:
        return 0.0
    return len(cand & orig) / max(len(cand), 1)


def rc_has_forbidden_new_citation(text, original):
    original_has_citation = bool(re.search(
        r'\b(Điều\s+\d+|Luật\s+\d+|Nghị định\s+\d+|Thông tư\s+\d+|Nghị quyết\s+\d+|\d+/\d{4}/[A-ZĐ-]+)\b',
        original, flags=re.IGNORECASE))
    if original_has_citation:
        return False
    return bool(re.search(r'\b(Điều\s+\d+|\d+/\d{4}/[A-ZĐ-]+)\b', str(text), flags=re.IGNORECASE))


def rc_validate_query_plan(raw_plan, question, raw_text=''):
    fallback = rc_fallback_query_plan(question)
    if not isinstance(raw_plan, dict):
        fallback['raw_plan_text'] = raw_text
        fallback['planner_error'] = 'planner returned non-dict'
        return fallback

    complexity = str(raw_plan.get('complexity', '')).strip().lower()
    if complexity not in COMPLEXITIES:
        complexity = fallback['complexity']

    qtype = str(raw_plan.get('question_type', '')).strip().lower()
    if qtype not in QUESTION_TYPES:
        qtype = fallback['question_type']

    atomic = rc_sanitize_list_of_strings(raw_plan.get('atomic_questions', []), max_items=RC_MAX_ATOMIC)
    clean_atomic = []
    for item in atomic:
        if rc_normalize_key(item) == rc_normalize_key(question):
            continue
        if rc_has_forbidden_new_citation(item, question):
            continue
        if rc_token_overlap_ratio(item, question) < 0.18:
            continue
        clean_atomic.append(item)

    if not clean_atomic and complexity in {'medium', 'complex'}:
        clean_atomic = fallback['atomic_questions']
    clean_atomic = clean_atomic[:RC_MAX_ATOMIC]

    terms = rc_sanitize_list_of_strings(raw_plan.get('must_have_terms', []), max_items=12, max_chars=60)
    terms = [t for t in terms if not rc_has_forbidden_new_citation(t, question)]
    if not terms:
        terms = fallback['must_have_terms']

    plan = {
        'complexity': complexity,
        'atomic_questions': clean_atomic,
        'must_have_terms': terms,
        'anchor_terms': rc_sanitize_list_of_strings(raw_plan.get('anchor_terms', []), max_items=12, max_chars=80),
        'legal_facets': rc_sanitize_list_of_strings(raw_plan.get('legal_facets', []), max_items=max(6, RC_MAX_FACETS), max_chars=220),
        'facet_profiles': rc_sanitize_facet_profiles(raw_plan.get('facet_profiles', []), max_items=max(6, RC_MAX_FACETS)),
        'domain_profile': raw_plan.get('domain_profile', {}) if isinstance(raw_plan.get('domain_profile', {}), dict) else {},
        'question_type': qtype,
        'rationale_short': rc_normalize_text(raw_plan.get('rationale_short', ''))[:260],
        'planner_fallback': False,
        'planner_error': '',
        'raw_plan_text': raw_text,
    }
    return rc_enrich_query_plan(question, rc_refine_query_plan_heuristics(question, plan))


def rc_llm_query_plan(question):
    # Use the existing llm_call (from cell 3b) to generate a query plan JSON.
    # Falls back to rule-based planning if llm_call is None or fails.
    question = rc_normalize_text(question)
    if llm_call is None:
        return rc_fallback_query_plan(question, reason='no_llm_call')
    system = (
        "Bạn là bộ phân tích truy vấn cho hệ thống truy hồi văn bản pháp luật Việt Nam. "
        "Chỉ phân rã ý hỏi, không trả lời câu hỏi, không suy đoán số điều, không bịa tên văn bản. "
        "Luôn trả về đúng một JSON object hợp lệ."
    )
    user = (
        "Câu hỏi:\n" + question + "\n\n"
        "Hãy trả về JSON theo schema:\n"
        '{"complexity": "simple|medium|complex", '
        '"atomic_questions": ["mệnh đề truy hồi độc lập, tối đa 4"], '
        '"must_have_terms": ["thuật ngữ bắt buộc lấy từ hoặc bám rất sát câu hỏi"], '
        '"anchor_terms": ["domain anchors"], '
        '"legal_facets": ["separate legal facets"], '
        '"facet_profiles": [{"facet_id": "stable_id", "label": "legal facet", '
        '"anchor_terms": [], "preferred_law_ids": [], "preferred_title_terms": [], '
        '"target_terms": [], "negative_terms": [], "priority": 1.0}], '
        '"domain_profile": {"labels": ["domain labels"]}, '
        '"question_type": "procedure|condition|rights_obligations|sanction|'
        'support_incentive|scenario|deadline|comparison|definition_listing|other", '
        '"rationale_short": "lý do ngắn"}\n\n'
        "Quy tắc:\n"
        "- Every atomic question must preserve specific domain anchors from the original.\n"
        "- Do not shorten into a generic form when the original contains a specific domain.\n"
        "- legal_facets should split legal aspects, not paraphrase the same question.\n"
        "- simple: 1 vấn đề, 1 văn bản/1 điều. medium: 2 ý. complex: nhiều mệnh đề, multi-hop.\n"
        "- Không nêu Điều X, mã luật nếu câu hỏi không nêu.\n"
        "- Không dùng markdown, không giải thích ngoài JSON."
    )
    prompt = system + "\n\n" + user
    try:
        raw = llm_call(prompt)
        plan_dict = rc_extract_json_object(raw)
        return rc_validate_query_plan(plan_dict, question, raw_text=raw)
    except Exception as e:
        plan = rc_fallback_query_plan(question, reason='llm_exception')
        plan['planner_error'] = repr(e)
        return plan


def rc_build_query_plan(question):
    # Main plan builder: LLM if available, rule-based fallback otherwise.
    question = rc_normalize_text(question)
    return rc_llm_query_plan(question)


# --- Query variants ---
def rc_must_terms_query(query_plan):
    terms = [rc_normalize_text(t) for t in query_plan.get('must_have_terms', []) if rc_normalize_text(t)]
    return ' '.join(terms[:16])


def rc_build_query_variants(question, query_plan=None):
    query_plan = query_plan or rc_fallback_query_plan(question)
    complexity = query_plan.get('complexity', 'medium')
    variants = []

    def add_variant(text, kind, weight, dense_only=False, lexical_only=False):
        text = rc_normalize_text(text)
        if not text:
            return
        key = rc_normalize_key(text)
        if key in {v['key'] for v in variants}:
            return
        variants.append({
            'text': text, 'kind': kind, 'weight': float(weight),
            'dense_only': bool(dense_only), 'lexical_only': bool(lexical_only),
            'key': key,
        })

    add_variant(question, 'original', 1.00)

    if complexity == 'simple':
        atomic_weights = [0.86]
        facet_weight = 0.00
        must_weight = 0.58
    elif complexity == 'complex':
        atomic_weights = [0.96, 0.93, 0.90, 0.87]
        facet_weight = 0.88
        must_weight = 0.68
    else:
        atomic_weights = [0.94, 0.90, 0.86]
        facet_weight = 0.91
        must_weight = 0.64

    for idx, atomic in enumerate(query_plan.get('atomic_questions', [])[:RC_MAX_ATOMIC]):
        weight = atomic_weights[min(idx, len(atomic_weights) - 1)]
        add_variant(atomic, 'atomic_' + str(idx + 1), weight)

    if complexity != 'simple':
        for idx, facet in enumerate(query_plan.get('facet_profiles', [])[:max(6, RC_MAX_FACETS)]):
            text = facet.get('query') or facet.get('label') or ''
            facet_id = re.sub(r'[^a-z0-9_]+', '_', rc_normalize_key(facet.get('facet_id', 'facet_' + str(idx + 1)))).strip('_') or ('facet_' + str(idx + 1))
            priority = float(facet.get('priority', 1.0) or 1.0)
            weight = min(0.97, facet_weight + 0.02 * max(priority - 1.0, 0.0))
            add_variant(text, 'facet_' + str(idx + 1) + '_' + facet_id, weight)

    if RC_USE_MUST_TERMS:
        term_query = rc_must_terms_query(query_plan)
        if term_query and rc_normalize_key(term_query) != rc_normalize_key(question):
            add_variant(term_query, 'must_terms', must_weight, lexical_only=True)

    return variants


# --- Weighted RRF ---
def rc_weighted_rrf_fuse(weighted_rankings, k=60, topk=96):
    scores = defaultdict(float)
    payload = {}
    sources = defaultdict(list)
    for ranking, weight, label in weighted_rankings:
        for rank, item in enumerate(ranking, start=1):
            row_idx = int(item['row_idx'])
            scores[row_idx] += float(weight) / (k + rank)
            payload.setdefault(row_idx, item)
            sources[row_idx].append({'source': label, 'rank': rank, 'weight': float(weight)})
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:topk]
    out = []
    for row_idx, score in fused:
        item = dict(payload[row_idx])
        item['row_idx'] = row_idx
        item['rrf_score'] = float(score)
        item['source_hits'] = sources[row_idx][:12]
        out.append(item)
    return out


# --- Article key helpers ---
def rc_article_key(c):
    law_id = str(c.get('law_id', '')).strip()
    ten = str(c.get('ten_van_ban', '')).strip()
    dieu = str(c.get('dieu_so', '')).strip()
    if law_id and ten and dieu:
        return law_id + '|' + ten + '|' + dieu
    return ''


def rc_canonical_article_key(c):
    law_id = str(c.get('law_id', '')).strip()
    dieu = str(c.get('dieu_so', '')).strip()
    if law_id and dieu:
        return law_id + '|' + dieu
    return rc_article_key(c)


# --- Domain scoring ---
def rc_score_candidate_domain(candidate, query_plan=None):
    query_plan = query_plan or {}
    profile = query_plan.get('domain_profile', {}) or {}
    law_id = str(candidate.get('law_id', '')).strip()
    title = str(candidate.get('ten_van_ban', ''))
    hay = rc_anchor_plain(' '.join([law_id, title, str(candidate.get('dieu_so', ''))]))
    score = 0.0
    reasons = []

    preferred_law_ids = set(profile.get('preferred_law_ids', []))
    soft_negative_law_ids = set(profile.get('soft_negative_law_ids', []))
    if law_id in preferred_law_ids:
        score += 0.75
        reasons.append('preferred_law_id')
    if law_id in soft_negative_law_ids:
        score -= 0.70
        reasons.append('soft_negative_law_id')

    for term in profile.get('preferred_title_terms', []):
        if rc_anchor_plain(term) in hay:
            score += 0.25
            reasons.append('preferred_title:' + term)
            break

    for term in profile.get('negative_title_terms', []):
        if rc_anchor_plain(term) in hay and law_id not in preferred_law_ids:
            score -= 0.80
            reasons.append('negative_title:' + term)
            break

    labels = set(profile.get('labels', []))
    if 'copyright_assessment' in labels and ('chuyen giao cong nghe' in hay or 'thuong mai' in hay) and law_id not in preferred_law_ids:
        score -= 0.55
        reasons.append('assessment_domain_drift')
    if 'copyright_software' in labels and 'so huu cong nghiep' in hay and law_id not in preferred_law_ids:
        score -= 0.80
        reasons.append('industrial_property_drift')
    if not reasons:
        reasons.append('neutral')
    return float(score), reasons[:6]


# --- Facet scoring ---
def rc_candidate_facet_haystack(candidate):
    pieces = [
        str(candidate.get('law_id', '')),
        str(candidate.get('ten_van_ban', '')),
        str(candidate.get('dieu_so', '')),
        str(candidate.get('article_key', '')),
        str(candidate.get('chunk_text', ''))[:2400],
    ]
    return rc_anchor_plain(' '.join(pieces))


def rc_score_one_facet(candidate, facet, support_variants=None):
    support_variants = support_variants or []
    law_id = str(candidate.get('law_id', '')).strip()
    hay = rc_candidate_facet_haystack(candidate)
    facet_id = str(facet.get('facet_id', '')).strip()
    rank_score = 0.0
    evidence_score = 0.0
    reasons = []
    preferred_law_ids = set(str(x).strip() for x in facet.get('preferred_law_ids', []) if str(x).strip())
    if law_id and law_id in preferred_law_ids:
        rank_score += 1.10
        reasons.append('preferred_law_id:' + law_id)
    for term in facet.get('preferred_title_terms', []):
        if rc_anchor_plain(term) in hay:
            rank_score += 0.20
            reasons.append('preferred_title:' + term)
            break
    anchor_hits = [term for term in facet.get('anchor_terms', []) if rc_anchor_plain(term) in hay]
    if anchor_hits:
        rank_score += min(0.30, 0.08 * len(set(rc_anchor_plain(x) for x in anchor_hits)))
        evidence_score += min(0.22, 0.06 * len(set(rc_anchor_plain(x) for x in anchor_hits)))
        reasons.append('anchor_terms:' + ','.join(anchor_hits[:4]))
    target_hits = [term for term in facet.get('target_terms', []) if rc_anchor_plain(term) in hay]
    if target_hits:
        rank_score += min(0.90, 0.18 * len(set(rc_anchor_plain(x) for x in target_hits)))
        evidence_score += min(1.25, 0.30 * len(set(rc_anchor_plain(x) for x in target_hits)))
        reasons.append('target_terms:' + ','.join(target_hits[:4]))
    if facet_id and any(facet_id in str(v) for v in support_variants):
        rank_score += 0.45
        evidence_score += 0.55
        reasons.append('facet_variant_support')
    for term in facet.get('negative_terms', []):
        if rc_anchor_plain(term) in hay and law_id not in preferred_law_ids:
            rank_score -= 0.45
            evidence_score -= 0.25
            reasons.append('negative_term:' + term)
            break
    return float(rank_score), float(evidence_score), reasons[:6]


def rc_score_candidate_facets(candidate, query_plan=None, support_variants=None):
    query_plan = query_plan or {}
    support_variants = support_variants or []
    facets = query_plan.get('facet_profiles', []) or []
    if not facets:
        return 0.0, [], [], {}, {}
    matched = []
    reasons = []
    score_map = {}
    evidence_map = {}
    best = 0.0
    for facet in facets:
        facet_id = str(facet.get('facet_id', '')).strip()
        score, evidence_score, facet_reasons = rc_score_one_facet(candidate, facet, support_variants=support_variants)
        score_map[facet_id] = float(score)
        evidence_map[facet_id] = float(evidence_score)
        best = max(best, float(score))
        threshold = 0.45 if facet.get('preferred_law_ids') else 0.55
        if evidence_score >= threshold:
            matched.append(facet_id)
            reasons.append({'facet_id': facet_id, 'score': float(score), 'evidence_score': float(evidence_score), 'reasons': facet_reasons})
    return float(best), matched, reasons[:8], score_map, evidence_map


def rc_is_generic_variant_text(text, query_plan=None):
    profile = (query_plan or {}).get('domain_profile', {})
    try:
        return rc_is_generic_atomic_question(text, profile)
    except Exception:
        plain = rc_anchor_plain(text)
        return _rc_plain_has_any(plain, ['tai lieu', 'chung cu', 'ho so', 'don yeu cau', 'hop dong', 'tranh chap'])


# --- Article aggregation (support bonuses across variants) ---
def rc_aggregate_article_candidates(variant_results, query_plan=None):
    # Group reranked chunks by canonical article, then score each article with:
    #   best rerank + early_bonus + original_bonus + atomic_bonus + facet_bonus
    #   + support_bonus + rrf_bonus + domain_weight*domain_score
    #   + facet_weight*facet_score + generic_penalty
    groups = OrderedDict()
    for vr in variant_results:
        variant = vr['variant']
        kind = str(variant.get('kind', ''))
        weight = float(variant.get('weight', 1.0))
        for rank, c in enumerate(vr.get('reranked', []), start=1):
            full_key = rc_article_key(c)
            key = rc_canonical_article_key(c)
            if not key:
                continue
            if key not in groups:
                groups[key] = {
                    'canonical_article_key': key,
                    'article_key': full_key,
                    'law_id': str(c.get('law_id', '')).strip(),
                    'ten_van_ban': str(c.get('ten_van_ban', '')).strip(),
                    'dieu_so': str(c.get('dieu_so', '')).strip(),
                    'best_context': dict(c),
                    'best_rerank_score': float(c.get('rerank_score', -1e9)),
                    'best_rrf_score': float(c.get('rrf_score', 0.0)),
                    'best_variant_kind': kind,
                    'best_variant_rank': rank,
                    'support_count': 0,
                    'support_rrf_sum': 0.0,
                    'support_variants': set(),
                    'source_labels': [],
                    'per_variant': {},
                    'generic_atomic_support': False,
                    'row_idx': int(c.get('row_idx', -1)),
                }
            g = groups[key]
            score = float(c.get('rerank_score', -1e9))
            g['support_count'] += 1
            g['support_rrf_sum'] += float(c.get('rrf_score', 0.0))
            g['support_variants'].add(kind)
            g['source_labels'].extend(s.get('source', '') for s in c.get('source_hits', []))
            if str(kind).startswith('atomic_') and rc_is_generic_variant_text(variant.get('text', ''), query_plan):
                g['generic_atomic_support'] = True
            prev = g['per_variant'].get(kind)
            if prev is None or score > prev['best_rerank_score']:
                g['per_variant'][kind] = {
                    'best_rank': rank,
                    'best_rerank_score': score,
                    'best_rrf_score': float(c.get('rrf_score', 0.0)),
                    'weight': weight,
                }
            if score > g['best_rerank_score']:
                g['best_context'] = dict(c)
                g['best_rerank_score'] = score
                g['best_rrf_score'] = float(c.get('rrf_score', 0.0))
                g['best_variant_kind'] = kind
                g['best_variant_rank'] = rank
                g['article_key'] = full_key
                g['ten_van_ban'] = str(c.get('ten_van_ban', '')).strip()
                g['row_idx'] = int(c.get('row_idx', -1))

    candidates = []
    for g in groups.values():
        variants = sorted(v for v in g['support_variants'] if v)
        original_support = 'original' in variants
        atomic_coverage_count = len([v for v in variants if v.startswith('atomic_')])
        facet_support_count = len([v for v in variants if v.startswith('facet_')])
        early_bonus = 0.20 / max(int(g['best_variant_rank']), 1)
        original_bonus = 0.24 if original_support else 0.0
        atomic_bonus = 0.18 * min(atomic_coverage_count, 4)
        facet_bonus = 0.12 * min(facet_support_count, 4)
        support_bonus = 0.05 * math.log1p(g['support_count'])
        rrf_bonus = 0.22 * g['support_rrf_sum']
        weak_penalty = -0.20 if variants == ['must_terms'] else 0.0
        domain_score, domain_reasons = rc_score_candidate_domain(g['best_context'], query_plan)
        facet_score, matched_facets, facet_reasons, facet_scores, facet_evidence_scores = \
            rc_score_candidate_facets(g['best_context'], query_plan, support_variants=variants)
        complexity = (query_plan or {}).get('complexity', 'medium')
        domain_weight = 0.0 if complexity == 'simple' else (0.08 if complexity == 'medium' else 0.18)
        facet_weight = 0.0 if complexity == 'simple' else (0.34 if complexity == 'medium' else 0.40)
        generic_penalty = -0.35 if (g.get('generic_atomic_support') and not original_support
                                    and domain_score < 0.30 and facet_score < 0.90) else 0.0
        article_score_raw = float(
            g['best_rerank_score'] + early_bonus + original_bonus + atomic_bonus
            + facet_bonus + support_bonus + rrf_bonus + weak_penalty
        )
        article_score = float(
            article_score_raw + domain_weight * domain_score
            + facet_weight * facet_score + generic_penalty
        )
        c = dict(g['best_context'])
        c['canonical_article_key'] = g['canonical_article_key']
        c['article_key'] = g['article_key']
        c['article_score'] = article_score
        c['article_score_raw'] = article_score_raw
        c['domain_score'] = float(domain_score)
        c['domain_reasons'] = domain_reasons
        c['facet_score'] = float(facet_score)
        c['best_facet_score'] = float(facet_score)
        c['facet_scores'] = facet_scores
        c['facet_evidence_score'] = float(max(facet_evidence_scores.values()) if facet_evidence_scores else 0.0)
        c['facet_evidence_scores'] = facet_evidence_scores
        c['matched_facets'] = matched_facets
        c['facet_reasons'] = facet_reasons
        c['generic_atomic_support'] = bool(g.get('generic_atomic_support', False))
        c['support_count'] = g['support_count']
        c['support_rrf_sum'] = float(g['support_rrf_sum'])
        c['support_variants'] = variants
        c['source_kinds'] = variants
        c['source_labels'] = sorted(set(g['source_labels']))[:16]
        c['original_support'] = bool(original_support)
        c['atomic_coverage_count'] = int(atomic_coverage_count)
        c['facet_support_count'] = int(facet_support_count)
        c['best_variant_kind'] = g['best_variant_kind']
        c['best_variant_rank'] = int(g['best_variant_rank'])
        c['best_variant_rerank_score'] = float(g['best_rerank_score'])
        c['per_variant'] = g['per_variant']
        c['row_idx'] = int(g.get('row_idx', -1))
        candidates.append(c)
    candidates.sort(key=lambda x: x['article_score'], reverse=True)
    return candidates


# --- Complexity-adaptive article budgets ---
def rc_article_bounds_for_complexity(complexity):
    if complexity == 'simple':
        return {'min_k': 1, 'target_k': 1, 'max_k': RC_ARTICLE_MAX_SIMPLE}
    if complexity == 'complex':
        return {'min_k': 4, 'target_k': 10, 'max_k': RC_ARTICLE_MAX_COMPLEX}
    return {'min_k': 2, 'target_k': 4, 'max_k': RC_ARTICLE_MAX_MEDIUM}


# --- Selection helpers ---
def rc_article_law_family(cand):
    return str(cand.get('law_id', '')).strip()


def rc_add_selected_article(selected, seen, cand, reason, family_counts=None, max_per_family=None):
    key = cand.get('canonical_article_key') or rc_canonical_article_key(cand)
    if not key or key in seen:
        return False
    family = rc_article_law_family(cand)
    count_before = family_counts.get(family, 0) if family_counts is not None else 0
    if family_counts is not None and max_per_family is not None and count_before >= max_per_family:
        return False
    item = dict(cand)
    item.setdefault('selection_reasons', [])
    item['selection_reasons'] = list(item.get('selection_reasons', [])) + [reason]
    item['law_family_count_before_select'] = int(count_before)
    selected.append(item)
    seen.add(key)
    if family_counts is not None:
        family_counts[family] = count_before + 1
    return True


def rc_candidates_for_variant(article_candidates, kind):
    out = [c for c in article_candidates if kind in c.get('per_variant', {})]
    out.sort(key=lambda c: (
        c.get('per_variant', {}).get(kind, {}).get('best_rank', 10**9),
        -float(c.get('best_facet_score', 0.0)),
        -float(c.get('domain_score', 0.0)),
        -float(c.get('per_variant', {}).get(kind, {}).get('best_rerank_score', -1e9)),
    ))
    return out


def rc_facet_profiles_for_selection(query_plan):
    facets = list((query_plan or {}).get('facet_profiles', []) or [])
    facets.sort(key=lambda f: -float(f.get('priority', 1.0) or 1.0))
    return facets


def rc_candidate_allowed_for_selection(cand, complexity, selected_len, min_k, require_facet=False):
    domain_score = float(cand.get('domain_score', 0.0))
    facet_score = float(cand.get('best_facet_score', cand.get('facet_score', 0.0)))
    if complexity == 'simple':
        return True
    if require_facet and facet_score < 0.85:
        return False
    if domain_score <= -0.75 and not cand.get('original_support', False) and facet_score < 1.10 and selected_len >= min_k:
        return False
    if complexity == 'medium':
        return (domain_score > -1.00 or cand.get('original_support', False)
                or facet_score >= 0.85 or selected_len < min_k)
    return True


def rc_facet_score_for_candidate(cand, facet):
    facet_id = str(facet.get('facet_id', '')).strip()
    return float((cand.get('facet_scores') or {}).get(facet_id, 0.0))


def rc_facet_evidence_score_for_candidate(cand, facet):
    facet_id = str(facet.get('facet_id', '')).strip()
    return float((cand.get('facet_evidence_scores') or {}).get(facet_id, 0.0))


def rc_candidate_law_preferred_for_facet(cand, facet):
    return str(cand.get('law_id', '')).strip() in set(
        str(x).strip() for x in facet.get('preferred_law_ids', []) if str(x).strip())


def rc_candidate_matches_facet(cand, facet):
    evidence = rc_facet_evidence_score_for_candidate(cand, facet)
    threshold = 0.45 if facet.get('preferred_law_ids') else 0.55
    return evidence >= threshold


def rc_candidates_for_facet(article_candidates, facet):
    facet_id = str(facet.get('facet_id', '')).strip()
    out = []
    for cand in article_candidates:
        evidence = rc_facet_evidence_score_for_candidate(cand, facet)
        if rc_candidate_matches_facet(cand, facet) or (
                rc_candidate_law_preferred_for_facet(cand, facet) and evidence >= 0.30):
            out.append(cand)
    out.sort(key=lambda c: (
        -float(rc_candidate_law_preferred_for_facet(c, facet)),
        -rc_facet_evidence_score_for_candidate(c, facet),
        -rc_facet_score_for_candidate(c, facet),
        int(c.get('best_variant_rank', 10**9)),
        -float(c.get('article_score', 0.0)),
    ))
    return out


def rc_preferred_doc_rescue_candidates(article_candidates, query_plan):
    profile = query_plan.get('domain_profile', {}) if query_plan else {}
    preferred = set(profile.get('preferred_law_ids', []))
    if not preferred:
        return []
    out = [c for c in article_candidates if str(c.get('law_id', '')).strip() in preferred]
    out.sort(key=lambda c: (
        -float(c.get('best_facet_score', 0.0)),
        -float(c.get('domain_score', 0.0)),
        -float(c.get('article_score', 0.0)),
        int(c.get('best_variant_rank', 10**9)),
    ))
    return out


def rc_preferred_facet_candidate_exists(article_candidates, facet):
    preferred = set(str(x).strip() for x in facet.get('preferred_law_ids', []) if str(x).strip())
    if not preferred:
        return False
    return any(rc_candidate_law_preferred_for_facet(cand, facet) and rc_candidate_matches_facet(cand, facet)
               for cand in article_candidates)


def rc_selected_covers_facet(selected, facet, article_candidates=None):
    preferred = set(str(x).strip() for x in facet.get('preferred_law_ids', []) if str(x).strip())
    require_preferred = bool(preferred) and rc_preferred_facet_candidate_exists(article_candidates or [], facet)
    for cand in selected:
        if require_preferred and not rc_candidate_law_preferred_for_facet(cand, facet):
            continue
        if rc_candidate_matches_facet(cand, facet):
            return True
    return False


def rc_selected_covered_facets(selected, query_plan=None, article_candidates=None):
    if query_plan:
        covered = set()
        for facet in rc_facet_profiles_for_selection(query_plan):
            if rc_selected_covers_facet(selected, facet, article_candidates=article_candidates):
                covered.add(str(facet.get('facet_id', '')))
        return covered
    covered = set()
    for cand in selected:
        for facet_id in cand.get('matched_facets', []):
            covered.add(facet_id)
    return covered


def rc_low_value_duplicate_facet(cand, covered_facets):
    matched = set(cand.get('matched_facets', []))
    if not matched:
        return False
    if not matched.issubset(covered_facets):
        return False
    return float(cand.get('best_facet_score', 0.0)) < 1.20 and not cand.get('original_support', False)


def rc_select_facet_coverage(article_candidates, query_plan, complexity, selected, seen,
                             family_counts, max_k, min_k, coverage_debug):
    facets = rc_facet_profiles_for_selection(query_plan)
    for facet in facets:
        if len(selected) >= max_k:
            coverage_debug.append({'facet_id': facet.get('facet_id', ''), 'status': 'skipped', 'reason': 'max_k'})
            break
        facet_id = facet.get('facet_id', '')
        if rc_selected_covers_facet(selected, facet, article_candidates=article_candidates):
            coverage_debug.append({'facet_id': facet_id, 'status': 'covered_before', 'reason': 'already_covered'})
            continue
        added = False
        skip_reasons = []
        facet_candidates = rc_candidates_for_facet(article_candidates, facet)
        for allow_over_cap in ([False, True] if complexity == 'complex' else [True]):
            for cand in facet_candidates:
                key = cand.get('canonical_article_key') or rc_canonical_article_key(cand)
                if key in seen:
                    skip_reasons.append('seen')
                    continue
                if not rc_candidate_matches_facet(cand, facet):
                    skip_reasons.append('low_facet_evidence')
                    continue
                if not rc_candidate_allowed_for_selection(cand, complexity, len(selected), min_k, require_facet=True):
                    skip_reasons.append('domain_or_low_facet')
                    continue
                max_per_family = 2 if (complexity == 'complex' and not allow_over_cap) else None
                if rc_add_selected_article(selected, seen, cand, 'facet_coverage:' + facet_id,
                                           family_counts=family_counts, max_per_family=max_per_family):
                    coverage_debug.append({
                        'facet_id': facet_id, 'status': 'selected',
                        'article_key': cand.get('article_key', ''),
                        'law_id': cand.get('law_id', ''),
                        'dieu_so': cand.get('dieu_so', ''),
                        'facet_score': rc_facet_score_for_candidate(cand, facet),
                        'facet_evidence_score': rc_facet_evidence_score_for_candidate(cand, facet),
                        'over_family_cap': bool(allow_over_cap),
                    })
                    added = True
                    break
                skip_reasons.append('family_cap')
            if added:
                break
        if not added:
            coverage_debug.append({
                'facet_id': facet_id, 'status': 'missed',
                'candidate_count': len(facet_candidates),
                'skip_reasons': sorted(set(skip_reasons))[:6],
            })


def rc_select_article_contexts(article_candidates, query_plan, variants=None):
    complexity = query_plan.get('complexity', 'medium')
    bounds = rc_article_bounds_for_complexity(complexity)
    variants = variants or []
    if not article_candidates:
        return [], {'selected_k': 0, 'reason': 'no_candidates', 'complexity': complexity,
                     'coverage_debug': [], **bounds}

    max_k = min(bounds['max_k'], len(article_candidates))
    min_k = min(bounds['min_k'], max_k)
    selected, seen = [], set()
    family_counts = {}
    coverage_debug = []
    reason = 'facet_coverage_aware_v2'

    if complexity == 'simple':
        rc_add_selected_article(selected, seen, article_candidates[0], 'simple_top1')
        if len(article_candidates) > 1 and len(selected) < max_k:
            top = float(article_candidates[0].get('article_score', 0.0))
            second = float(article_candidates[1].get('article_score', 0.0))
            same_law = article_candidates[0].get('law_id') == article_candidates[1].get('law_id')
            if same_law and second >= top - 0.55:
                rc_add_selected_article(selected, seen, article_candidates[1], 'simple_close_same_law')
    else:
        rc_select_facet_coverage(article_candidates, query_plan, complexity, selected, seen,
                                  family_counts, max_k, min_k, coverage_debug)

        original_quota = 1 if complexity == 'medium' else 2
        for cand in rc_candidates_for_variant(article_candidates, 'original')[:original_quota]:
            if len(selected) >= max_k:
                break
            if rc_candidate_allowed_for_selection(cand, complexity, len(selected), min_k):
                rc_add_selected_article(selected, seen, cand, 'original_quota', family_counts=family_counts)

        if complexity == 'complex':
            rescue_added = 0
            for cand in rc_preferred_doc_rescue_candidates(article_candidates, query_plan):
                if len(selected) >= max_k or rescue_added >= 4:
                    break
                if not rc_candidate_allowed_for_selection(cand, complexity, len(selected), min_k):
                    continue
                if rc_low_value_duplicate_facet(cand, rc_selected_covered_facets(selected, query_plan, article_candidates)) and len(selected) >= min_k:
                    continue
                if rc_add_selected_article(selected, seen, cand, 'preferred_doc_rescue',
                                           family_counts=family_counts, max_per_family=2):
                    rescue_added += 1

        base_variant_quota = 1 if complexity == 'medium' else 2
        for v in variants:
            kind = str(v.get('kind', ''))
            if not (kind.startswith('atomic_') or kind.startswith('facet_')):
                continue
            added_for_variant = 0
            for cand in rc_candidates_for_variant(article_candidates, kind):
                if len(selected) >= max_k:
                    break
                if not rc_candidate_allowed_for_selection(cand, complexity, len(selected), min_k):
                    continue
                if rc_low_value_duplicate_facet(cand, rc_selected_covered_facets(selected, query_plan, article_candidates)) and len(selected) >= min_k:
                    continue
                if added_for_variant >= base_variant_quota:
                    if not (float(cand.get('best_facet_score', 0.0)) >= 1.10 or cand.get('original_support', False)):
                        continue
                max_per_family = 3 if complexity == 'complex' else None
                if rc_add_selected_article(selected, seen, cand, kind + '_quota',
                                           family_counts=family_counts, max_per_family=max_per_family):
                    added_for_variant += 1
                if added_for_variant >= base_variant_quota + 1:
                    break

        for cand in article_candidates:
            if len(selected) >= max_k:
                break
            if float(cand.get('domain_score', 0.0)) <= -0.75 and len(selected) >= min_k:
                continue
            if rc_low_value_duplicate_facet(cand, rc_selected_covered_facets(selected, query_plan, article_candidates)) and len(selected) >= min_k:
                continue
            if rc_candidate_allowed_for_selection(cand, complexity, len(selected), min_k):
                rc_add_selected_article(selected, seen, cand, 'score_fill',
                                        family_counts=family_counts,
                                        max_per_family=3 if complexity == 'complex' else None)

        if len(selected) < min_k:
            for cand in article_candidates:
                if len(selected) >= min_k or len(selected) >= max_k:
                    break
                rc_add_selected_article(selected, seen, cand, 'min_k_backfill', family_counts=family_counts)

    selected.sort(key=lambda x: x.get('article_score', 0.0), reverse=True)
    return selected, {
        'complexity': complexity,
        'selected_k': int(len(selected)),
        'reason': reason,
        **bounds,
        'covered_facets': sorted(rc_selected_covered_facets(selected, query_plan, article_candidates)),
        'coverage_debug': coverage_debug,
    }


# --- Build relevant lists (grader format) ---
def rc_make_relevant_lists(article_contexts):
    docs, articles = [], []
    seen_docs, seen_articles = set(), set()
    for c in article_contexts:
        law_id = str(c.get('law_id', '')).strip()
        ten = str(c.get('ten_van_ban', '')).strip()
        dieu = str(c.get('dieu_so', '')).strip()
        if law_id and ten:
            d = law_id + '|' + ten
            if d not in seen_docs:
                seen_docs.add(d)
                docs.append(d)
        if law_id and ten and dieu:
            a = law_id + '|' + ten + '|' + dieu
            if a not in seen_articles:
                seen_articles.add(a)
                articles.append(a)
    return docs, articles


# --- Lightweight result wrapper (RetrievalResult-compatible) ---
class RCResult:
    """Minimal result object with relevant_docs()/relevant_articles() methods
    so it is a drop-in for RetrievalResult in the smoke test and batch cells."""

    def __init__(self, query, query_plan, relevant_docs, relevant_articles,
                 selected_articles, selection_debug):
        self.query = query
        self.query_plan = query_plan
        self._relevant_docs = relevant_docs
        self._relevant_articles = relevant_articles
        self.selected_articles = selected_articles
        self.selection_debug = selection_debug
        self.routing = None  # not used by the grader record

    def relevant_articles(self):
        return self._relevant_articles

    def relevant_docs(self):
        return self._relevant_docs


# --- Main entry point ---
def retrieve_recall_first(query):
    """Recall-first retrieval, ported from kaggle-hybridrag-decomp-anchor-v2.

    Reuses the modular pipeline's callables (lexical_search, dense_search,
    reranker, text_provider). Falls back to pipeline.retrieve(query) on any
    exception so the batch never dies.
    """
    try:
        q = rc_normalize_text(query)
        query_plan = rc_build_query_plan(q)
        variants = rc_build_query_variants(q, query_plan)

        # Pull callables off the modular pipeline object.
        lexical = pipeline.lexical_search
        dense = pipeline.dense_search
        reranker = pipeline.reranker
        text_provider = pipeline.text_provider

        per_variant_topk = max(RC_RERANK_TOPK, RC_CANDIDATE_TOPK // max(len(variants), 1))
        variant_results = []

        for variant in variants:
            kind = str(variant.get('kind', ''))
            vtext = variant['text']

            # --- Lexical (BM25) ---
            lex_hits = []
            if not variant.get('dense_only'):
                lex_hits = lexical(vtext, RC_BM25_TOPK) or []

            # --- Dense (FAISS) ---
            d_hits = []
            if not variant.get('lexical_only') and dense is not None:
                d_hits = dense(vtext, RC_DENSE_TOPK) or []

            # --- Weighted RRF per variant ---
            local_rankings = []
            if lex_hits:
                local_rankings.append((lex_hits, variant['weight'], 'lexical:' + kind))
            if d_hits:
                local_rankings.append((d_hits, variant['weight'], 'dense:' + kind))
            local_fused = rc_weighted_rrf_fuse(local_rankings, k=RC_RRF_K, topk=per_variant_topk)

            # --- Fetch text for reranking ---
            row_idxs = [int(h['row_idx']) for h in local_fused]
            texts = text_provider(row_idxs) if (text_provider and row_idxs) else {}

            # Build candidate dicts with chunk_text for facet scoring + reranking.
            contexts = []
            for h in local_fused:
                ridx = int(h['row_idx'])
                txt = (texts.get(ridx, '') or '')[:2400]
                c = dict(h)
                c['row_idx'] = ridx
                c['chunk_text'] = txt
                contexts.append(c)

            # --- Per-variant rerank (cross-encoder on the VARIANT text) ---
            reranked = []
            if reranker is not None and contexts:
                passages = [(c.get('chunk_text', '') or '')[:RC_RERANK_TRUNCATE] for c in contexts]
                scores = reranker(vtext, passages) or []
                for c, s in zip(contexts, scores):
                    c['rerank_score'] = float(s)
                    reranked.append(c)
                reranked.sort(key=lambda x: x['rerank_score'], reverse=True)
                reranked = reranked[:RC_RERANK_TOPK]
            else:
                # No reranker: keep fused order, use rrf_score as a proxy.
                for c in contexts:
                    c['rerank_score'] = float(c.get('rrf_score', 0.0))
                reranked = contexts[:RC_RERANK_TOPK]

            variant_results.append({
                'variant': variant,
                'lex_hits': lex_hits,
                'dense_hits': d_hits,
                'fused': local_fused,
                'reranked': reranked,
            })

        # --- Aggregate across variants (support bonuses + domain/facet scores) ---
        article_candidates = rc_aggregate_article_candidates(variant_results, query_plan=query_plan)

        # --- Facet-coverage selection (complexity-adaptive budgets) ---
        selected_articles, selection_debug = rc_select_article_contexts(
            article_candidates, query_plan, variants=variants)

        relevant_docs, relevant_articles = rc_make_relevant_lists(selected_articles)

        return RCResult(
            query=q, query_plan=query_plan,
            relevant_docs=relevant_docs,
            relevant_articles=relevant_articles,
            selected_articles=selected_articles,
            selection_debug=selection_debug,
        )
    except Exception as e:
        # Safety net: never kill the batch. Fall back to the modular pipeline.
        print('⚠ retrieve_recall_first error (%s); falling back to pipeline.retrieve()' % repr(e))
        return pipeline.retrieve(query)


# --- Batch-level retrieve() dispatcher ---
def retrieve(query):
    """Dispatch to recall-first or modular pipeline based on USE_RECALL_FIRST."""
    if USE_RECALL_FIRST:
        return retrieve_recall_first(query)
    return pipeline.retrieve(query)


print('✓ Recall-first enhancer ready (USE_RECALL_FIRST=%s)' % USE_RECALL_FIRST)
if USE_RECALL_FIRST:
    print('  retrieve(query) -> retrieve_recall_first() [multi-variant + facet-coverage]')
else:
    print('  retrieve(query) -> pipeline.retrieve() [modular 7-stage]')


In [ ]:
# ============================================================================
# 5. Load the competition test set (2000 questions)
# ============================================================================
# A JSON list of {"id", "question"} records. Set TEST_PATH to override.
TEST_PATH = BASE / 'data/stage6_data/R2AIStage1DATA.json'

assert TEST_PATH.exists(), (
    f"Test set not found: {TEST_PATH}\n"
    "Upload R2AIStage1DATA.json to data/stage6_data/ on Drive."
)
with open(TEST_PATH, encoding='utf-8') as f:
    questions = json.load(f)

# Normalise: ensure every record has id + question; ids kept as given by the
# test set (the grader matches them verbatim). Coerce numeric strings to int
# only when they round-trip, else keep the original value.
def _norm_id(v):
    try:
        iv = int(v)
        return iv if str(iv) == str(v).strip() else v
    except (TypeError, ValueError):
        return v

questions = [
    {'id': _norm_id(r.get('id')), 'question': (r.get('question') or '').strip()}
    for r in questions
]
# Drop any malformed rows (missing id or empty question).
questions = [q for q in questions if q['id'] is not None and q['question']]

print(f"✓ Loaded {len(questions)} questions from {TEST_PATH.name}")
print(f"  id range: {questions[0]['id']} … {questions[-1]['id']}")


In [ ]:
# ============================================================================
# 5b. Smoke test: run ONE question end-to-end and print the results.json record
# ============================================================================
import json, time

# Pick the first question from the loaded test set (or set SMOKE_ID to a
# specific id). Verifies the pipeline works and the record matches the grader
# schema before committing to the full 2000-question batch.
SMOKE_ID = None  # e.g. 5 to use question id 5; None = first question

if SMOKE_ID is not None:
    _smoke_item = next((q for q in questions if q['id'] == SMOKE_ID), None)
    assert _smoke_item is not None, f"No question with id={SMOKE_ID}"
else:
    _smoke_item = questions[0]

_qid, _qtxt = _smoke_item['id'], _smoke_item['question']
print(f"Question id={_qid}")
print(f"Q: {_qtxt}\n")

_t0 = time.time()
_result = retrieve(_qtxt)
_elapsed = time.time() - _t0

_record = {
    'id': _qid,
    'question': _qtxt,
    'answer': '',
    'relevant_docs': _result.relevant_docs(),
    'relevant_articles': _result.relevant_articles(),
}

print(f"Retrieved in {_elapsed:.2f}s")
print(f"relevant_docs ({len(_record['relevant_docs'])}):")
for d in _record['relevant_docs']:
    print(f"  - {d}")
print(f"relevant_articles ({len(_record['relevant_articles'])}):")
for a in _record['relevant_articles']:
    print(f"  - {a}")

print("\n--- results.json record ---")
print(json.dumps(_record, ensure_ascii=False, indent=2))

# Quick schema sanity (same checks as the validate cell).
assert set(_record) == {'id', 'question', 'answer', 'relevant_docs', 'relevant_articles'}
assert all('|' in d for d in _record['relevant_docs']), "relevant_docs must contain '|'"
assert all(a.count('|') >= 2 for a in _record['relevant_articles']), "relevant_articles must have >= 2 '|'"

# Final-select stage rendered in the agent_instruction/final_select_prompt.md
# JSON shape: selected_articles (scored + reasoned) + supporting_chunks + notes.
print("\n--- final_select_prompt.json ---")
try:
    print(json.dumps(final_select_prompt_json(_result), ensure_ascii=False, indent=2))
except Exception:
    print('(final_select_prompt_json skipped — RCResult has no modular selection)')
print("\n✓ Smoke test passed — record matches grader schema. Safe to run the batch cell.")


In [ ]:
# ============================================================================
# 6. Batch run: 2000 questions -> 4 shards in parallel (4 threads, 1 GPU)
# ============================================================================
# One shared embed_model + faiss_gpu + reranker + pipeline (loaded once in
# cell 4). 4 worker threads each take a strided 500-question shard; GPU calls
# are lock-guarded (cell 4) so only one kernel runs at a time, but the GIL is
# released during GPU wait, letting the other 3 threads run CPU stages (BM25,
# graph expand, RRF, final select). Expected ~2-3x on one L4/A100.
#
# CRASH-SAFE + RESUMABLE (the point of this cell):
#   * Each record is appended to OUT_DIR/results.shardN.jsonl and flushed
#     IMMEDIATELY after retrieval -- no 50-record buffer to lose on interrupt.
#   * On (re)start, each shard reads its existing .jsonl, collects the ids
#     already done, and SKIPS them. A partial last line (killed mid-write) is
#     detected and truncated, so that record is re-run cleanly -- no dupes,
#     no incomplete records.
#   * So: Colab disconnect / runtime restart / OOM kill -> just re-run this
#     cell; it resumes from where it stopped. Delete the .jsonl files (or set
#     RESUME=False) to force a clean full run.
import os, time, json, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

RESULTS_PATH        = BASE / 'results.json'
SUBMISSION_ZIP_PATH = BASE / 'submission.zip'
N_SHARDS            = 4          # parallel parts; set 2 if GPU mem is tight

# Where the per-shard .jsonl progress files live. Default = a subfolder of
# BASE (Drive) so they survive a Colab runtime reset. Point this anywhere you
# want the in-progress data written immediately.
OUT_DIR             = BASE / 'batch_out'
RESUME              = True       # False = ignore existing .jsonl, start over

OUT_DIR.mkdir(parents=True, exist_ok=True)

def _shard_path(n):
    return OUT_DIR / f'results.shard{n}.jsonl'

def _load_done_ids(path):
    """Return (done_ids:set, good_bytes:int) from an existing .jsonl.

    Reads line by line; on the first un-parseable line, stops and reports the
    byte offset of the end of the last GOOD line -- the caller truncates there
    so the partial record is re-run. Survives a kill mid-write with no dupes.
    """
    done = set()
    if not path.exists():
        return done, 0
    good_bytes = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                rec = json.loads(line)
            except (ValueError, TypeError):
                break  # partial last line -- stop here; caller truncates
            if isinstance(rec, dict) and rec.get('id') is not None:
                done.add(rec['id'])
            good_bytes += len(line.encode('utf-8'))
    return done, good_bytes

def _process_shard(shard_id, items):
    path = _shard_path(shard_id)

    # --- Resume: skip already-done ids, trim a partial trailing line --------
    if RESUME:
        done_ids, good_bytes = _load_done_ids(path)
        if path.exists() and good_bytes < path.stat().st_size:
            # Truncate the partial last line so that record is re-run cleanly.
            with open(path, 'r+b') as fb:
                fb.truncate(good_bytes)
            print(f"[shard {shard_id}] truncated partial trailing line "
                  f"({path.stat().st_size - good_bytes}B)", flush=True)
    else:
        done_ids = set()
        if path.exists():
            path.unlink()  # fresh start

    n_done_before = len(done_ids)
    todo = [it for it in items if it['id'] not in done_ids]
    if n_done_before:
        print(f"[shard {shard_id}] resuming: {n_done_before} done, "
              f"{len(todo)} remaining", flush=True)

    # Append mode + per-record flush: every record hits disk before we move on.
    f = open(path, 'a', encoding='utf-8')
    t0 = time.time()
    pbar = tqdm(todo, desc=f'shard {shard_id}', position=shard_id,
                leave=True, dynamic_ncols=True)
    try:
        for i, item in enumerate(pbar):
            qid, qtxt = item['id'], item['question']
            try:
                result = retrieve(qtxt)
                record = {
                    'id': qid, 'question': qtxt, 'answer': '',
                    'relevant_docs': result.relevant_docs(),
                    'relevant_articles': result.relevant_articles(),
                }
            except Exception as e:
                print(f"[shard {shard_id}] error id={qid}: {e}")
                record = {'id': qid, 'question': qtxt, 'answer': '',
                          'relevant_docs': [], 'relevant_articles': []}
            # Immediate, durable-enough write: one JSON per line + flush.
            # flush() pushes to the OS page cache (survives a process kill /
            # Colab disconnect). fsync would also survive power loss but is
            # very slow on Drive; per-record flush is the right trade-off here.
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
            f.flush()
            new_done = i + 1
            total_done = n_done_before + new_done
            el = time.time() - t0
            pbar.set_postfix(rate=f'{new_done/el:.2f}q/s' if el > 0 else '-',
                             eta=f'{el/new_done*(len(todo)-new_done)/60:.1f}m' if new_done else '-',
                             done=total_done,
                             refresh=False)
    finally:
        f.close()
        pbar.close()
    return shard_id, path

# Split questions into N_SHARDS strided shards (id coverage preserved).
shards = [questions[i::N_SHARDS] for i in range(N_SHARDS)]
print(f"Running {len(questions)} questions across {N_SHARDS} threads "
      f"(~{len(shards[0])}/shard). Output -> {OUT_DIR}", flush=True)

start_time = time.time()
shard_paths = {}
with ThreadPoolExecutor(max_workers=N_SHARDS) as ex:
    futs = [ex.submit(_process_shard, n, sh) for n, sh in enumerate(shards)]
    for fut in as_completed(futs):
        sid, p = fut.result()
        shard_paths[sid] = p
        print(f"  + shard {sid} done -> {p}", flush=True)

total_time = time.time() - start_time
print(f"\n+ All shards done in {total_time/60:.1f} min "
      f"({total_time/len(questions):.2f}s/question across {N_SHARDS} threads)")
print(f"  Shard files: {[str(_shard_path(n)) for n in range(N_SHARDS)]}")
print("  Run the next cell to merge shards -> results.json.")


In [ ]:
# ============================================================================
# 6b. Merge the shard .jsonl files -> results.json (by id, ordered as loaded)
# ============================================================================
# Reads OUT_DIR/results.shard{0..N_SHARDS-1}.jsonl (one JSON record per line,
# written incrementally by the batch cell). Also tolerates legacy
# results.shardN.json array files for backward compatibility. Dedupes by id
# (last wins), then reorders to match the original `questions` order so id
# coverage is exact. The grader's validate + package cells run on results.json.
import json, os
from pathlib import Path

def _iter_shard_records(p):
    """Yield records from a shard file. Handles .jsonl (line-per-record) and
    legacy .json (a JSON array). Skips blank / un-parseable lines."""
    if not p.exists():
        return
    txt = p.read_text(encoding='utf-8')
    if txt.lstrip().startswith('['):
        # Legacy array form (results.shardN.json).
        try:
            for r in json.loads(txt):
                yield r
        except (ValueError, TypeError) as e:
            print(f"! {p.name}: legacy array parse failed ({e})")
        return
    # JSONL form (results.shardN.jsonl).
    for line in txt.splitlines():
        if not line.strip():
            continue
        try:
            yield json.loads(line)
        except (ValueError, TypeError):
            # Skip a partial trailing line (shouldn't happen after the batch
            # cell's truncation, but be defensive).
            continue

merged = {}
for n in range(N_SHARDS):
    p = _shard_path(n)
    if not p.exists():
        print(f"! missing {p.name} -- was shard {n} started?")
        continue
    n_rec = 0
    for r in _iter_shard_records(p):
        merged[r['id']] = r
        n_rec += 1
    print(f"  shard {n}: {n_rec} records from {p.name}")

# Order exactly as the loaded test set; fill any gap with an empty-valid record.
records = []
missing = 0
for q in questions:
    if q['id'] in merged:
        records.append(merged[q['id']])
    else:
        missing += 1
        records.append({'id': q['id'], 'question': q['question'], 'answer': '',
                        'relevant_docs': [], 'relevant_articles': []})
if missing:
    print(f"! {missing} ids missing from all shards -- emitted empty records")

tmp = str(RESULTS_PATH) + '.tmp'
with open(tmp, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
os.replace(tmp, RESULTS_PATH)

print(f"+ Merged {len(records)} records -> {RESULTS_PATH}")
print(f"  id coverage: {len(records)}/{len(questions)} "
      f"(unique ids: {len({r['id'] for r in records})})")


In [ ]:
# ============================================================================
# 7. Validate results.json (grader contract: array, schema, id coverage)
# ============================================================================
data = json.loads(Path(RESULTS_PATH).read_text(encoding='utf-8'))

assert isinstance(data, list), "results.json must be a JSON array"
assert len(data) == len(questions), (
    f"record count mismatch: got {len(data)}, expected {len(questions)}"
)

REQUIRED = ['id', 'question', 'answer', 'relevant_docs', 'relevant_articles']
for r in data:
    assert all(k in r for k in REQUIRED), f"record missing fields: {r.keys()}"

got_ids = {r['id'] for r in data}
exp_ids = {q['id'] for q in questions}
assert got_ids == exp_ids, (
    f"id coverage mismatch — missing: {sorted(exp_ids - got_ids)[:5]}…"
)

# Format checks: relevant_docs are 'law|ten', relevant_articles are 'law|ten|Điều'.
bad_docs = sum(1 for r in data for d in r['relevant_docs'] if d.count('|') < 1)
bad_arts = sum(1 for r in data for a in r['relevant_articles'] if a.count('|') < 2)
print(f"✓ results.json valid: {len(data)} records, schema + id coverage OK")
if bad_docs or bad_arts:
    print(f"  ⚠ format warnings: bad_docs={bad_docs}, bad_articles={bad_arts} (check separator '|')")


In [ ]:
# ============================================================================
# 8. Package submission.zip (results.json at zip root) + backup to Drive
# ============================================================================
import zipfile

with zipfile.ZipFile(SUBMISSION_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(RESULTS_PATH, arcname='results.json')

print(f"Results    : {RESULTS_PATH}")
print(f"Submission : {SUBMISSION_ZIP_PATH}  (results.json at zip root)")
print("Copy to your local machine or keep in Drive.")
